# __`Social Data Analysis`__ 

__Generalities, Complex Networks and Node-Centric Metrics__

In this practical work, we will first check  :
 - if a network satisfies the 3 constraints to be considered as a complex network and 
 - then, we will determine how important nodes are within this network. 

The library we will use for handling networks is networkx

### __`Install the expecting libraries`__ 
(not necessary if you are certain these libs are installed on your system)

In [2]:
#%pip install --upgrade pip
#%pip -q install --upgrade pip
#%pip -q install pandas
#%pip -q install networkx
#%pip -q install ipython-cypher
#%pip -q install py2neo
#%pip -q install neo4j
#%pip -q install matplotlib
#%pip -q install neo4j-viz
#%pip -q install graphdatascience
#%pip -q install palettable
#%pip -q install nbimporter

### __`Import the useful packages`__     
You can avoid the first line if you are not using a Jupyter notebook. This line enables the visualization to be displayed in the notebook.

In [3]:
import os
import pandas as pd
from neo4j import GraphDatabase
from neo4j_viz.gds import from_gds
from graphdatascience import GraphDataScience

## __`8 Pipeline`__

:::


### __`8.1 Filtrage type I démarche`__

__Création du Graphe Projeté (neo4j_userInfluence_filtered) :__

 - __Type de Projection__ : Nous utilisons une projection Cypher, ce qui offre une grande flexibilité pour définir la structure du graphe GDS.
 - __Nœuds Projetés__ : 
    - Cela signifie que le graphe GDS (original_graph_name) ne contiendra que des nœuds de type User. 
    - Aucun nœud Tweet ou Event n'est explicitement projeté comme entité nodale dans ce graphe GDS.

 - __Relations Projetées__ : 
    - La requête de relations est complexe et définit des liens synthétiques entre les utilisateurs :

      - La relation dans le graphe GDS ira de u2 (l'utilisateur qui interagit/amplifie) vers u1 (l'utilisateur qui a posté le contenu original).

    - Des propriétés sont créées sur ces relations synthétiques : 
      - interactionWeight (combien de fois ce type d'interaction u2 -> u1 via ce mécanisme a eu lieu pour un tweet original donné), 
      - topic (le topicId du tweet original), et originalTweetId (l'ID du tweet original au centre de cette interaction).
      - Paramètres : Le eventTypeIdParamValue est utilisé pour cibler l'analyse.

:::

:::

#### __` 8.1.1 Création du graphe`__ 
 
`Graphe projeté avec des nœuds de type User et des relations synthétiques`

 - __Propriétés du Graphe__ : 
    - Le graphe projeté est stocké sous le nom neo4j_userInfluence_filtered.
    - Les propriétés des nœuds sont stockées dans le graphe projeté.
    - Les propriétés des relations sont stockées dans le graphe projeté.

:::    

In [4]:
neo4j_userInfluence_filtered = """
    CALL gds.graph.project.cypher(
    $graphName,
    'MATCH (u:User) RETURN id(u) AS id', 
    'WITH $eventTypeIdParam AS targetEventType
     MATCH (u1:User)-[:POSTED]->(t_original:Tweet)-[:IS_ABOUT]->(e:Event),
           (u2:User)-[:POSTED]->(t_interaction:Tweet)-[:RETWEETED|REPLY_TO]->(t_original)      
     WHERE t_original.topicId = e.topicId
       AND e.eventTypeId = targetEventType
     RETURN id(u1) AS source,         
            id(u2) AS target,
            count(*) AS interactionWeight,        
            t_original.topicId AS topic, 
            t_original.id AS originalTweetId,
            e.eventTypeId AS eventTypeId',
        {
          readConcurrency: 4,
          parameters: { eventTypeIdParam: $eventTypeIdParamValue }  
        }
  ) YIELD graphName, nodeCount, relationshipCount
"""  
#list_graphs(gds) 

:::

### __` 8.1.2 Extraction des propriétés des noeuds`__

__`Extraction des propriétés des noeuds`__ : 
 - __Propriétés__ : userId, eventTypeId, pagerank, community, embedding
 - __Propriétés des Nœuds__ : stockées dans le graphe projeté
 - __Propriétés des Relations__ : stockées dans le graphe projeté

::: 

In [5]:
# Code PRÉCÉDENT pour les propriétés des nœuds (supposons qu'il est exécuté)
query_nodeProperties = """
    CALL gds.graph.nodeProperties.stream(
        $graphName, 
        $nodeProperties, 
         $nodeLabels,
        { 
          listNodeLabels: true 
        })
    YIELD nodeId AS gdsNodeId, 
        nodeLabels AS gdsNodeLabelsInProjection, 
        nodeProperty, 
        propertyValue
    WITH gds.util.asNode(gdsNodeId) AS originalNode, 
        gdsNodeId, gdsNodeLabelsInProjection, 
        nodeProperty, 
        propertyValue
    RETURN
        originalNode,
        elementId(originalNode) AS originalId,
        labels(originalNode) AS originalLabels,
        gdsNodeId,
        gdsNodeLabelsInProjection,
        nodeProperty,
        propertyValue
    ORDER BY originalId, nodeProperty"""

:::

### __` 8.1.3 Extraction des propriétés issues des relations`__

__`Extraction des Propriétés des Relations (extract_all_relationship_properties) :`__

 - Cette fonction récupère les informations de base des relations (IDs source/cible, type, noms des nœuds source/cible via gds.util.asNode).
 - Ensuite, pour chaque nom de propriété spécifié (properties_to_extract : topic, interactionWeight, originalTweetId),

    - Elle exécute gds.graph.relationshipProperty.stream et fusionne les résultats. 
    - C'est une manière de collecter toutes les propriétés des relations projetées.

:::    

In [6]:

def extract_all_relationship_properties(gds, graph_name, property_names=[]):
    #if not property_names:return None

    query_base = f"""
        CALL gds.graph.relationships.stream('{graph_name}')
        YIELD sourceNodeId, targetNodeId, relationshipType
        WITH sourceNodeId, targetNodeId, relationshipType,
             gds.util.asNode(sourceNodeId) AS sourceGdsNode,
             gds.util.asNode(targetNodeId) AS targetGdsNode
        RETURN
            sourceNodeId, targetNodeId,sourceGdsNode, targetGdsNode,
            coalesce(sourceGdsNode.name, "") AS sourceName,
            coalesce(targetGdsNode.name, "") AS targetName,
            labels(sourceGdsNode) AS sourceLabels,
            elementId(sourceGdsNode) AS sourceElementId, 
        CASE 
        WHEN sourceGdsNode:User THEN sourceGdsNode.name
        ELSE sourceGdsNode.text 
        END AS sourceNameOrText,
                relationshipType,
            labels(targetGdsNode) AS targetLabels,
            elementId(targetGdsNode) AS targetElementId,
        CASE
        WHEN targetGdsNode:User  THEN targetGdsNode.name
        ELSE targetGdsNode.text 
        END AS targetNodeOrText,
        // Combinaison et gestion des hashtags
        CASE
        WHEN sourceGdsNode:Tweet AND targetGdsNode:Tweet THEN
            apoc.coll.union(
            [match IN apoc.text.regexGroups(sourceGdsNode.text, '#(\\\\w+)') | match[1]],
            [match IN apoc.text.regexGroups(targetGdsNode.text, '#(\\\\w+)') | match[1]]
            )
        WHEN sourceGdsNode:Tweet THEN [match IN apoc.text.regexGroups(sourceGdsNode.text, '#(\\\\w+)') | match[1]]
        WHEN targetGdsNode:Tweet THEN [match IN apoc.text.regexGroups(targetGdsNode.text, '#(\\\\w+)') | match[1]]
        ELSE null
        END AS Hashtags
            """
    base_df = gds.run_cypher(query_base)
    merged_df = base_df.copy()

    if not property_names:
        return merged_df
    for prop_name in property_names:
        query_prop = f"""
            CALL   gds.graph.relationshipProperty.stream('{graph_name}' , '{prop_name}' )
            YIELD  sourceNodeId, targetNodeId, relationshipType, propertyValue AS `{prop_name}`
            RETURN sourceNodeId, targetNodeId, relationshipType, `{prop_name}`
            """
        result_df_prop = gds.run_cypher(query_prop)
        merged_df = pd.merge(merged_df, result_df_prop,on=['sourceNodeId', 'targetNodeId', 'relationshipType'], how='left')
        desired_columns = ['sourceElementId','targetElementId','sourceGdsNode','targetGdsNode','sourceNodeId','targetNodeId', 'sourceLabel', 'targetLabel','sourceName', 'targetName']
        desired_columns.extend(property_names)
    
    final_columns = [col for col in desired_columns if col in merged_df.columns]
    merged_df_final = merged_df[final_columns]
    return merged_df_final

### __` 8.1.4 Mise en oeuvre de la démarche`__

In [20]:

import nbimporter
from tp1_3_m2_2 import get_gdsConnection, drop_wholeGraph

gds=get_gdsConnection()
drop_wholeGraph(gds)

2.13.4


NameError: name 'query_inspect_graph' is not defined

:::

#### __` 8.1.4.1 Ìnstantiation du Graphe Projeté`__ 
La projection s'effectue sur l'ensemble des éléme,ts du graphe qui satisfont la relation  

     MATCH (u1:User)-[:POSTED]->(t_original:Tweet)-[:IS_ABOUT]->(e:Event),
           (u2:User)-[:POSTED]->(t_interaction:Tweet)-[:RETWEETED|REPLY_TO]->(t_original)      
     WHERE t_original.topicId = e.topicId
       AND e.eventTypeId = targetEventType

graphe projeté est instancié avec les paramètres suivants :

 - __Nom du Graphe Projeté__ : neo4j_userInfluence_filtered
 - __eventTypeId__ : 5 # filtrage sur les événements de type 5
 - __Type de Graphe__ : User
 - __Type de Projection__ : Cypher
 - __Nœuds Projetés__ : User
 - __Relations Projetées__ : Synthétiques entre utilisateurs (u2 -> u1)
 - __Propriétés des Nœuds__ : stockées dans le graphe projeté
 - __Propriétés des Relations__ : stockées dans le graphe projeté
 - __Propriétés des Relations__ : stockées dans le graphe projeté

 ::: 

In [ ]:
eventTypeId = 5
original_graph_name = f'userInfluence{eventTypeId}'
properties_to_extract = ['topic', 'interactionWeight','originalTweetId' ]
relationshipTypes = ['__ALL__']
nodeLabels = ['__ALL__']

gds=get_gdsConnection()
# Créer le graphe projeté
gds.graph.drop(original_graph_name)
out=gds.run_cypher(neo4j_userInfluence_filtered, params= {'graphName': original_graph_name, 'eventTypeIdParamValue': eventTypeId})

::: 

#### __` 8.1.4.2 Calcul du pageRank`__ 

Le PageRank (userInfluencePagerank) est calculé sur ce graphe original_graph_name. 

:::

In [ ]:
# Créer le pargeRank property sur le graphe projeté
clean_projectionGraph_Parameter(gds, original_graph_name, 'userInfluencePagerank')
node_pagerank_df = gds.run_cypher(query_pageRank_mutate, params={
                                    'graphName': original_graph_name, 
                                    'nodeProperties': 'userInfluencePagerank', 
                                    'nodeLabels': nodeLabels, 
                                    'relationshipTypes': relationshipTypes })

:::

#### __` 8.1.4.3 Vectorisation des noeuds`__ : 

Les algorithmes d'intégration de nœuds calculent des représentations vectorielles de faible dimension des nœuds d'un graphe. Ces vecteurs, également appelés intégrations, peuvent être utilisés pour l'apprentissage automatique. 

 - __Type de Graphe__ : Le graphe projeté est de type User, ce qui signifie que nous ne calculons la vectorisation que pour les nœuds de type User.
 - __Type d'Algorithme__ : Nous utilisons l'algorithme FastRP pour la vectorisation des nœuds. 
 - __Paramètres__ : 
    - embeddingDimension = 128 (dimensionnalité de l'espace d'embedding)
    - walkLength = 10 (longueur de la marche aléatoire)
    - iterations = 20 (nombre d'itérations)
    - randomSeed = 42 (graine aléatoire pour la reproductibilité)

:::     

In [ ]:
# Créer le fastrp_embedding property sur le graphe projeté
clean_projectionGraph_Parameter(gds, original_graph_name, 'fastrp_embedding')
node_embeddings_df = gds.run_cypher(query_mutate_embeddings,params={
                                    'graphName': original_graph_name, 
                                    'mutateProperty': 'fastrp_embedding', 
                                    'nodeLabels': nodeLabels, 
                                    'relationshipTypes': relationshipTypes})

::: 
#### __` 8.1.4.4 Calcul de la Propagation des Labels`__ :

 - La propagation de labels (community) est calculée.

::: 

In [ ]:
# Créer le community property sur le graphe projeté
clean_projectionGraph_Parameter(gds, original_graph_name, 'community')
labelPropagation = gds.run_cypher(query_labelPropagation_mutate, params={
                                    'graphName': original_graph_name, 
                                    'community': 'community'})

:::

#### __` 8.1.4.5 Extraction des propriétés des relations`__ : 
 - __Propriétés__ : topic, interactionWeight, originalTweetId
 - __Propriétés des Nœuds__ : stockées dans le graphe projeté
 - __Propriétés des Relations__ : stockées dans le graphe projeté   

::: 

::: 

`Chaque enregistrement du dataFrame représente une relation unique dans le graphe projeté qui correspond aux critères de projection de notre requête pour laquelle les propriétés de relation (topic, interactionWeight ...) sont extraites.`

:::

In [ ]:
# Appeler la fonction d'extraction
relationship_details_df = extract_all_relationship_properties(gds, original_graph_name, properties_to_extract)
df_alt_replaced = relationship_details_df.replace({'topic': dico_topicId})

#### __` 8.1.4.6 Extraction des propriétés des noeuds`__ : 

In [ ]:
nodeProperties = ['userInfluencePagerank', 'community', 'fastrp_embedding']
node_dfprop = gds.run_cypher(query_nodeProperties, params={'graphName': original_graph_name, 'nodeLabels': nodeLabels, 'nodeProperties': nodeProperties})
df_pivot = node_dfprop.pivot_table(index='originalId', columns='nodeProperty', values='propertyValue', aggfunc='first').reset_index()
df_pivot.columns.name = None  # Supprimer le nom de l'index des colonnes
df_pivot_for = df_pivot.copy() # Garder une copie pour éviter des problèmes futurs

In [ ]:
df_pivot_for

### __` 8.1.5 Réorganisation des données`__ 


::: 

#### __` 8.1.5.1 Fusion de type inner (Jointure interne)`__    

C'est la méthode la plus directe pour ne conserver que les lignes où une correspondance existe entre les deux DataFrames.
 - Les lignes qui n'ont pas de correspondance dans l'autre DataFrame sont supprimées.
 - Cela garantit que seules les lignes avec des relations valides entre les deux DataFrames sont conservées.    

:::

In [ ]:
# Renommer les colonnes de df_pivot AVANT de fusionner
df_pivot_for_source = df_pivot.rename(columns={
    'community': 'source_community',
    'fastrp_embedding': 'source_fastrp_embedding',
    'userInfluencePagerank': 'source_userInfluencePagerank',
    'originalId': 'source_originalId'  # Renommer aussi originalId
})

df_pivot_for_target = df_pivot.rename(columns={
    'community': 'target_community',
    'fastrp_embedding': 'target_fastrp_embedding',
    'userInfluencePagerank': 'target_userInfluencePagerank',
    'originalId': 'target_originalId' # Renommer aussi originalId
})    

::: 
#### __` 8.1.5.2 Réconciliation des données`__ : 
 - __Propriétés__ : userId, eventTypeId, pagerank, community, embedding
 - __Propriétés des Nœuds__ : stockées dans le graphe projeté
 - __Propriétés des Relations__ : stockées dans le graphe projeté

:::

In [ ]:
df_alt_replaced

In [ ]:

final_reconciled_df_source = pd.merge(
    df_alt_replaced,
    df_pivot_for_source,
    left_on='sourceElementId', 
    right_on='source_originalId',  
    how='left'
)

final_reconciled_df_target = pd.merge(
    df_alt_replaced,
    df_pivot_for_target,
    left_on='targetElementId', 
    right_on='target_originalId',  
    how='left'
)

final_reconciled_df = pd.merge(
    final_reconciled_df_source,
    df_pivot_for_target,
    left_on='targetElementId',
    right_on='target_originalId',
    how='left',
    suffixes=('_source', '_target')
)

:::   
#### __` 8.1.5.3 Enrichissement des Données et Formatage Final`__

 - Les propriétés des relations extraites (relationship_details_df) sont enrichies avec les scores PageRank des nœuds source et cible.
 - Certaines colonnes sont supprimées pour obtenir le DataFrame.

:::

In [ ]:
final_reconciled_df.drop(columns=['sourceElementId','targetElementId','sourceGdsNode','targetGdsNode', 'sourceNodeId', 'targetNodeId'], axis=0)

### __` 8.1.6 Représentation des graphes`__ 

#### __` 8.1.6.1 Représentation du pageRank`__ 

In [ ]:
# Vérifie si le graphe existe si oui il est effacé de la mémoire
sampling_ratio = 0.2
sampled_graph_name = f'{original_graph_name}_sampled'
nodeProperties = 'userInfluencePagerank'

gds.graph.drop(sampled_graph_name)
# Effectue une réduction d'échelle pour autorisezr l'affichage de la projection
result=gds.run_cypher( query_random_walk, 
                params={'subgraph_name': original_graph_name,'sampled_graph_name': sampled_graph_name ,
                        'sampling_ratio': sampling_ratio,'nodeProperties': nodeProperties,})

# Rendu avec paramètres de performance
VG = display_intermediate(gds,sampled_graph_name,nodeProperties)
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

#### __` 8.1.6.2 Représentation des communauté`__ 

In [ ]:
# Vérifie si le graphe existe si oui il est effacé de la mémoire
sampling_ratio = 0.2
sampled_graph_name = f'{original_graph_name}_sampled'
nodeProperties = 'community'

gds.graph.drop(sampled_graph_name)
# Effectue une réduction d'échelle pour autorisezr l'affichage de la projection
result=gds.run_cypher( query_random_walk, 
                params={'subgraph_name': original_graph_name,'sampled_graph_name': sampled_graph_name ,
                        'sampling_ratio': sampling_ratio,'nodeProperties': nodeProperties,})

# Rendu avec paramètres de performance
VG = display_intermediate(gds,sampled_graph_name,nodeProperties)
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

__`La visualisation est pilotée par la propriété nodale community`__  

 - Ce graphique montre un échantillon du réseau d'influence où les utilisateurs sont colorés par leur ID de communauté. 

 - Structure Générale du Graphe (disposition "Force-Directed") utilise une disposition où les nœuds (représentés par des points) sont positionnés comme s'ils se repoussaient mutuellement, tandis que les liens (non explicitement visibles ici, mais influençant la position) les tirent ensemble. Cela tend à regrouper les nœuds qui sont plus densément connectés.

 - La dispersion des couleurs et la structure suggèrent que l'algorithme de détection de communauté a trouvé de nombreuses petites communautés (probablement une par utilisateur), indiquant une influence fragmentée plutôt que de grands blocs cohésifs. Les nœuds centraux (plus denses, et les "blobs" lumineux) représentent le cœur de l'activité d'influence, tandis que les nœuds périphériques sont moins intégrés.

 - Si l'on s'attendait à voir des "îlots" de quelques couleurs distinctes représentant de grandes communautés, le résultat actuel suggère que la structure du graphe ou les paramètres de l'algorithme de communauté n'ont pas conduit à ce type de regroupement




## __`8.2 Pipeline type II démarche`__

### __`8.2.1 Création du Graphe Principal (query_create_main_graph)`__    

 - Nous commençons par projeter un graphe en mémoire avec la Graph Data Science Library (GDS) de Neo4j. 
 - Ce graphe, inclut différents types de nœuds avec leurs propriétés spécifiées, et les relations qui les lient.

In [ ]:
# Création du graphe projeté principal 
drop_wholeGraph(gds)
graphName = 'graphName'

properties_Event=['eventTypeId','topicId']
properties_User=['friends_count', 'tweets_count','listed_count', 'statuses_count','favourites_count', 'followers_count', 'id']
properties_Tweet=['retweet_count','annotation_num_judgements','favorite_count','id','topicId']
properties_Hashtag=['occurences']
properties_PostCategory=[]

properties = [properties_Event, properties_User, properties_Tweet, properties_Hashtag, properties_PostCategory]

orientation= ['NATURAL','NATURAL','NATURAL','NATURAL','NATURAL','NATURAL','NATURAL','NATURAL','NATURAL','NATURAL']

gds.graph.drop('graphName')
out=gds.run_cypher(query_create_main_graph, params={'graphName': 'graphName', 'properties': properties, 'orientation': orientation})

### __`8.2.2 Filtrage du Graphe`__ 

 - À partir du graphe principal, nous extrayons un sous-graphe filtré pour se concentrer sur un eventTypeId spécifique (ici, 5).    
 - Le filtre inclut tous les nœuds User et Tweet, ainsi que des types de relations spécifiques (POSTED, RETWEETS, MENTIONS, REPLIED_TO).    
 - Ceci permet de conserver le contexte autour de l'événement

In [ ]:
# Filtrage à partir du graphe projeté principal 
eventTypeFilter = """
        CALL gds.graph.filter(
            $filteredName,     
            $sourceName,       
            $nodeFilterPredicate, 
            $relFilterPredicate,   
            { parameters: { eventId: $eventIdValue } } 
        ) YIELD graphName, fromGraphName, nodeCount, relationshipCount
        RETURN graphName, fromGraphName, nodeCount, relationshipCount
        """ 

### __`8.2.2.1 Création du graphe projeté filtré`__

Nous projetons un graphe filtré (neo4j_userInfluence_filtered) qui ne contient Les nœuds Event, User et Tweet ainsi que les relations associées.  

In [ ]:
gds=get_gdsConnection()
# Nom du graphe initial
original_graph_name = 'graphName'
event_type_id_to_filter = 5 # L'EventType ID souhaité
filtered_graph_name = f'filteredGraph_EventType{event_type_id_to_filter}'
sampled_graph_name = f'userInfluenceGraph{event_type_id_to_filter}_sampled'
sampling_ratio = 0.01

relationship = ['MENTIONS', 'POSTED', 'REPLIED_TO', 'RETWEETS']
nodeLabels = ['Event', 'Tweet', 'User']

node_filter = f"(n:Event AND n.eventTypeId = $eventId ) OR n:User OR n:Tweet"
relationship_filter = "r:POSTED OR r:RETWEETS OR r:MENTIONS OR r:REPLIED_TO"

# Supprimer les graphes projetés précédemment
gds.graph.drop(filtered_graph_name, False)
# Appeler gds.graph.filter
filter_result = gds.run_cypher(
                eventTypeFilter,
                params={'filteredName': filtered_graph_name,'sourceName': original_graph_name,
                        'nodeFilterPredicate': node_filter,'relFilterPredicate': relationship_filter,
                        'eventIdValue': event_type_id_to_filter })

### __`8.2.3 Calcul du pageRank`__

 - A partir de ce graphe filtré, nous exécutons l'algorithme PageRank. 
 - L'option mutate écrit les scores PageRank directement sur les nœuds du graphe en mémoire GDS sous la propriété userInfluencePagerank.    
 - userInfluencePagerank mesure l'influence relative des nœuds Event, Tweet, et User au sein de ce contexte filtré.

In [ ]:
# Copie la propriété mutate sur le graphe projeté
clean_projectionGraph_Parameter(gds, filtered_graph_name, 'InfluencePagerank')
out = gds.run_cypher(query_pageRank_mutate, params={'graphName': filtered_graph_name, 
                                              'nodeProperties': 'InfluencePagerank', 
                                              'nodeLabels': nodeLabels, 
                                              'relationshipTypes': relationship })

### __`8.2.4 Calcul de la vectorisation des noeuds`__

In [ ]:
# Créer le fastrp_embedding property sur le graphe projeté
clean_projectionGraph_Parameter(gds, filtered_graph_name, 'fastrp_embedding')
out = gds.run_cypher(query_mutate_embeddings,params={'graphName': filtered_graph_name, 
                                            'mutateProperty': 'fastrp_embedding', 
                                            'nodeLabels': nodeLabels, 
                                            'relationshipTypes': relationship})

### __`8.2.5 Calcul des communautés des noeuds`__

In [ ]:
# Créer le community property sur le graphe projeté
clean_projectionGraph_Parameter(gds, filtered_graph_name, 'community')
labelPropagation = gds.run_cypher(query_labelPropagation_mutate, params={'graphName': filtered_graph_name, 'community': 'community'})

### __`8.2.6 Extraction des propriétés des nœuds`__

 - Nous effectuons ensuite l'extraction les propriétés des nœuds Tweet et User séparément à partir du sampled_graph_name.        
 - Les données extraites (format long) sont pivotées pour que chaque nodeProperty devienne une colonne (format large).     
 - Cela donne df_pivot_tweet et df_pivot_user.   
 - Nous utilisons un dictionnaire dico_topicId pour remplacer les identifiants bruts des topicId par leurs libéllés, pour les tweets.

#### __`8.2.6.1 Déclinaison des valeurs pour les Tweets`__

In [ ]:
properties_Tweet=['retweet_count','annotation_num_judgements','favorite_count','topicId','InfluencePagerank','community','fastrp_embedding']
nodes_df_tweet= gds.run_cypher(query_nodeProperties, params={'graphName': filtered_graph_name , 'nodeLabels':['Tweet'] ,'nodeProperties': properties_Tweet})
df_pivot_tweet = nodes_df_tweet.pivot(index='originalId', columns=['nodeProperty'], values='propertyValue')
df_pivot_tweet = df_pivot_tweet.reset_index()
df_pivot_tweet = df_pivot_tweet.replace({'topicId': dico_topicId})

In [ ]:
df_pivot_tweet

#### __`8.2.6.2 Déclinaison des valeurs pour les Users`__

In [ ]:
properties_User=['friends_count', 'tweets_count','listed_count', 'statuses_count','favourites_count', 'followers_count','InfluencePagerank','community','fastrp_embedding']
nodes_df_user= gds.run_cypher(query_nodeProperties, params={'graphName': filtered_graph_name , 'nodeLabels':['User'] ,'nodeProperties': properties_User})
df_pivot_user = nodes_df_user.pivot(index='originalId', columns=['nodeProperty'], values='propertyValue')
df_pivot_user = df_pivot_user.reset_index()
df_pivot_user = df_pivot_user.replace({'topicId': dico_topicId})

In [ ]:
df_pivot_user

In [ ]:
# Si originalId est l'index, le transformer en colonne

if df_pivot_user.index.name == 'originalId':
    print("Resetting index for df_pivot_user")
    df_pivot_user_for_merge = df_pivot_user.reset_index()
else:
    df_pivot_user_for_merge = df_pivot_user.copy() # Assurez-vous qu'originalId est une colonne
    print("originalId is not the index, copying df_pivot_user")

if df_pivot_tweet.index.name == 'originalId':
    print("originalId is the index, resetting index for df_pivot_tweet")
    df_pivot_tweet_for_merge = df_pivot_tweet.reset_index()
else:
    df_pivot_tweet_for_merge = df_pivot_tweet.copy() # Assurez-vous qu'originalId est une colonne
    print("originalId is not the index, copying df_pivot_tweet")


### __`8.2.7 Extraction des Relations (query_base_relationships)`__ 

 - Nous effectuons l'extraction les relations à partir du graphe sampled_graph_name.      
 - La fonction gds.util.asNode() est utilisée pour accéder aux propriétés des nœuds source et cible 
 - Ces informations sont accessibles à partir des ID GDS internes, y compris leur id original (Neo4j) et d'autres propriétés comme name ou text.

In [ ]:
relationship_details_df = extract_all_relationship_properties(gds, filtered_graph_name)
relationship_details_df

### __`8.2.8 Préparation des Données de Nœuds pour la Fusion`__ :    

Les DataFrames pivotés pour les tweets et les utilisateurs (df_pivot_tweet_renamed, df_pivot_user_renamed) sont préparés :
 - La colonne id (l'ID original Neo4j) est renommée en originalId.
 - userInfluencePagerank est renommé en pageRankScore.
 - Une colonne gdsNodeLabels est ajoutée manuellement.
 - Les colonnes pertinentes (originalId, pageRankScore, gdsNodeLabels, topicId) sont sélectionnées.

Ces deux DataFrames sont concaténés en nodes_wide_df_combined, puis copiés dans node_info_to_merge qui servira de table de consultation.

In [ ]:
# Fusion 1: Joindre df_relations avec les propriétés des utilisateurs (nœuds source)
# Cela ajoute les propriétés de l'utilisateur source à chaque relation.
df_merged_rel_users = pd.merge(
    relationship_details_df,
    df_pivot_user_for_merge,
    left_on='sourceElementId',
    right_on='originalId',  # 'originalId' de df_pivot_user_for_merge
    how='inner',             # 'inner' pour ne garder que les relations où l'utilisateur source existe dans df_pivot_user_for_merge
                             # Vous pouvez utiliser 'left' pour garder toutes les relations de df_relations.
    suffixes=('_relation', '_user') # Suffixes pour les colonnes en double (ex: originalId_relation, originalId_user)
)

In [ ]:
df_merged_rel_users

#### __`8.2.8.1 Réconciliation des Données (Fusion)`__     

 - Le DataFrame des relations (relationships_df) est fusionné avec node_info_to_merge. 
 - Cette opération est effectuée une première fois pour enrichir les relations avec les informations du nœud source.     
 - La jointure se fait sur sourceOriginalId (de relationships_df) et originalId (de node_info_to_merge).     
 - Les colonnes récupérées (pageRankScore_lookup, gdsNodeLabels_lookup, et implicitement topicId de node_info_to_merge) sont ajoutées et renommées pour indiquer qu'elles concernent la source (ex: sourcePageRankScore).

In [ ]:
# Fusion 2: Joindre le résultat précédent avec les propriétés des tweets (nœuds cible)
# Cela ajoute les propriétés du tweet cible à chaque relation (qui a déjà les infos de l'utilisateur source).
df_reconciled_final = pd.merge(
    df_merged_rel_users,
    df_pivot_tweet_for_merge,
    left_on='targetElementId', # ID cible de la relation
    right_on='originalId',    # 'originalId' de df_pivot_tweet_for_merge
    how='inner',               # 'inner' pour ne garder que les relations où le tweet cible existe.
                               # Vous pouvez utiliser 'left' pour garder toutes les lignes de df_merged_rel_users.
    suffixes=('_user', '_tweet') # Suffixes pour les colonnes en double (ex: InfluencePagerank_user, InfluencePagerank_tweet)
)

df_reconciled_final = df_reconciled_final.drop(columns=['targetNodeId','sourceNodeId','originalId_user', 'originalId_tweet','sourceOriginalId','targetOriginalId',''], errors='ignore')
df_reconciled_final

# __`9 Exploitation des résultats`__

::: 

## __`9.1 plusieurs façons d'exploiter ce fichier`__ :


 - __Identifier les Acteurs et Contenus les Plus Influents__ :

   - __Utilisateurs influents__ : Le tri du DataFrame l'un ou l'autre des scores de PageRanking de manière décroissante
     - sourcePageRankScore (si la source est un utilisateur)
     - targetPageRankScore (si la cible est un utilisateur) 

    Les utilisateurs avec les scores les plus élevés sont considérés comme les plus influents dans le contexte de l'événement analysé et des types de relations considérées (MENTIONS, POSTED, REPLIED_TO, RETWEETS).

     - __Action__ : Effectuons l'extraction de sourceElementId (ou targetElementId) et le sourceName (ou targetName) des utilisateurs ayant les scores PageRank les plus hauts.

   - __Tweets influents__: De même, si les Tweets ont un score PageRank (ce qui est le cas car userInfluencePagerank était calculé pour les nœuds Tweet), nous pouvons identifier les tweets qui ont joué un rôle central dans la diffusion d'information ou la génération d'engagement autour de l'événement.

:::

:::

### __`9.1.1 Échantillonnage du Graphe (query_random_walk)`__  

 - Pour faciliter la visualisation et potentiellement accélérer les analyses ultérieures, nous créeons un échantillon plus petit.         
 - Cette échantillon (sampled_graph_name) est issu du graphe filtré en utilisant une méthode de marche aléatoire (Random Walk).    
 - Les propriétés des nœuds, y compris userInfluencePagerank, sont conservées dans cet échantillon.  

:::

In [ ]:
nodeProperties = 'InfluencePagerank'
gds.graph.drop(sampled_graph_name)
result=gds.run_cypher( query_random_walk, 
                params={'subgraph_name': filtered_graph_name,'sampled_graph_name': sampled_graph_name ,
                        'sampling_ratio': sampling_ratio,'nodeProperties': nodeProperties,})

In [ ]:
list_graphs(gds)

::: 
### __`9.1.2 Visualisation InfluencePagerank`__  

 - Nous visualisons ce graphe échantillonné. 
 - L'image montre un réseau où les nœuds (points) sont connectés par des liens. 
 - La taille et/ou la couleur des nœuds sont probablement influencées par leur score userInfluencePagerank, mettant en évidence les nœuds les plus influents
 :::

In [ ]:
# Rendu avec paramètres de performance
VG = display_intermediate(gds,sampled_graph_name,nodeProperties)
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

:::
### __`9.1.3 Visualisation community`__ :   

 - Nous visualisons ce graphe échantillonné. 
 - L'image montre un réseau où les nœuds (points) sont connectés par des liens. 
 - La taille et/ou la couleur des nœuds sont probablement influencées par leur score userInfluencePagerank, mettant en évidence les nœuds les plus influents
::: 

In [ ]:
nodeProperties = 'community'
gds.graph.drop(sampled_graph_name)
result=gds.run_cypher( query_random_walk, 
                params={'subgraph_name': filtered_graph_name,'sampled_graph_name': sampled_graph_name ,
                        'sampling_ratio': sampling_ratio,'nodeProperties': nodeProperties,})

# Rendu avec paramètres de performance
VG = display_intermediate(gds,sampled_graph_name,nodeProperties)
VG.render( width="100%", height="600px", max_allowed_nodes=11000)
#list_graphs(gds)

::: 
### __`9.3 Comparaison des deux Pipelines`__ 

|Caractéristique |	Pipeline 2 |Pipeline 1  |
|:-:             |---               |---    |	
|__Type de Projection__ |Principalement Native GDS (gds.graph.project), puis filtrage.|Projection Cypher GDS (gds.graph.project.cypher).|
|__Nœuds dans GDS__|Initialement :Event,User,Tweet,Hashtag,PostCategory. Après filtrage :Event(spécifique),User,Tweet.|	UniquementUser.|
|__Relations dans GDS__|Relations directes et existantes dans Neo4j (ex:POSTED,MENTIONS,RETWEETS), filtrées.|__Relations synthétiques__ User -> Usercréées par la requête Cypher, basées sur des chaînes d'interaction impliquant des Tweets et des Events. Les Tweets et Events ne sont pas des nœuds dans ce graphe GDS.|
|__Propriétés des Relations__|Principalement les propriétés existantes des relations.|Nouvelles propriétés calculées sur les relations synthétiques:interactionWeight,topic,originalTweetId.|
|__Calcul du PageRank__|Sur un graphe hétérogène (Users, Tweets, Events) reflétant les interactions directes.|Sur un graphe homogène (Useruniquement)où les liens représentent une influence indirecte via des interactions avec des tweets liés à un événement.|
|__Focalisation de l'Analyse__|Influence générale et connectivité au sein d'un écosystème autour d'un événement (qui parle à qui, quel tweet est central).|Influence inter-utilisateurs spécifique: Qui amplifie le contenu de qui, ou qui est dont le contenu est amplifié, dans le contexte précis d'uneventTypeIdet d'untopicIdcommun.|
|__Complexité de Projection__|Projection initiale plus simple, mais nécessitait plus de manipulation post-GDS pour combiner différents types de nœuds.|Projection initiale plus complexe (la requête Cypher définit la logique du graphe), mais peut cibler plus précisément la structure d'analyse souhaitée.|
|__Type d'Influence Mesurée__|Plutôt une centralité générale dans le discours autour de l'événement.|Une mesure de l'influence en tant qu'initiateur de discussions(êtreu1) ou en tant qu'amplificateur/participant clé(êtreu2) dans ces discussions.|
:::

:::

__Pipeline 1__ :
 - Est le meilleur pour analyser une forme spécifique d'influence entre utilisateurs, médiatisée par des tweets liés à un événement.
 - Identifier les "faiseurs d'opinion":
  
    - (u1 dont les tweets sont souvent repris/discutés) et les "amplificateurs clés" 
    - (u2 qui reprennent/discutent activement ces tweets).
 - Quantifier la force de ces liens d'influence indirecte (via interactionWeight).
 - Comprendre quels topicId et originalTweetId catalysent le plus cette influence inter-utilisateurs.

:::

:::

__Pipeline 2__ :

 - Est le meilleur pour comprendre la structure globale des interactions autour d'un événement, impliquant différents types d'entités.
 - Identifier les tweets les plus centraux ou les utilisateurs les plus connectés de manière générale dans ce contexte.
 - Analyser différents types de relations directes.

::: 

:::

__`Conclusion pour le Pipeline 1`__ :

Le pipeline 1 est une approche très ciblée et sophistiquée pour modéliser un type particulier d'influence.       
Le fait que sourcePageRank et targetPageRank soient parfois identiques pour les utilisateurs les plus influents est une conséquence directe de la manière dont le graphe User-User est construit : 

 - Ces utilisateurs sont centraux à la fois comme initiateurs et comme interacteurs (parfois avec leur propre contenu, ou leur contenu est si central qu'ils apparaissent dans les deux rôles pour différentes interactions). Cela met en lumière les acteurs véritablement dominants dans la dynamique d'interaction que vous avez modélisée.

:::

 
__Projection__ `gds.graph.project Directed & Undirected graph, Heterogenous nodes`
- Objectif : Charger le graphe en mémoire.
- Analyse : Etape initiale de l'analyse avec GDS permettant de définir la topologie et les propriétés des nœuds/relations.

__PageRank__ `gds.pageRank Directed & Undirected graph, Heterogenous nodes`
- Objectif : Calculer l'influence des nœuds.
- Analyse : Très utile pour obtenir une première mesure de l'importance des nœuds. C'est une étape pour identifier les acteurs clés. 

__Embedding__ `gds.fastRP représentations vectorielles denses des nœuds`  
- Objectif : Représentation des nœuds dans un espace vectoriel.
- Analyse : Permet de capturer la structure du graphe et de faciliter les tâches d'apprentissage automatique ou de visualisation.   

__Échantillonnage__ `gds.graph.sample.cnarw Directed & Undirected graph, Heterogenous nodes`
- Objectif : Réduire la taille du graphe tout en préservant ses propriétés structurelles.
- Analyse : Utile pour travailler avec de grands graphes, en conservant les relations essentielles.

__Détection de Communautés Undirected graph__    
- `gds.leiden Undirected graph, Heterogenous nodes`
- `gds.localClusteringCoefficient Undirected graph, Heterogenous nodes`

__Détection de Communautés  Directed & Undirected graph__  
- `gds.kmeans Directed & Undirected graph, Heterogenous nodes`
- `gds.labelPropagation Directed & Undirected graph, Heterogenous nodes`
- `gds.modularity.stats Directed & Undirected graph, Heterogenous nodes`
- `gds.articulationPoints Directed & Undirected graph, Heterogenous nodes`

__Centralité__
- `gds.degree, Identifier les nœuds centraux, Directed graph, Heterogenous nodes`
- `gds.closeness, Identifier les nœuds centraux, Directed graph, Heterogenous nodes`
- `gds.betweenness, Évaluer l'importance des nœuds, Directed graph, Heterogenous nodes`
- `gds.eigenvector, Étudier les propriétés des communautés, Directed & Undirected graph, Heterogenous nodes`

__Recommandations__ `gds.alpha.ml`
- Objectif : Créer des systèmes de recommandation.      
- Analyse : Permet de suggérer des éléments basés sur les relations entre les nœuds.
- Utilisation : Recommandations de contenu, d'utilisateurs, etc.

Similarité `gds.nodeSimilarity`
- Objectif : Identifier les nœuds similaires.
- Analyse : Utile pour trouver des "doublons" ou des nœuds ayant des rôles similaires.
- Utilisation : Recommandations, détection de communautés, etc.

__GraphSAGE__ `gds.graphSAGE`
- Objectif : Effectuer des analyses de graphes (typiquement, générer des embeddings inductifs).
- Analyse : GraphSAGE est un autre algorithme d'embedding puissant, souvent utilisé pour des graphes dynamiques ou pour généraliser à des nœuds non vus. Il pourrait être une alternative ou un complément à FastRP. Son placement en fin de liste suggère une exploration plus avancée.


__Fondations__ : Projection, PageRank pour une compréhension de base.   
__Préparation pour ML/Analyse Avancée__ : Embeddings.   
__Gestion de la Taille/Focus__ : Échantillonnage.   
__Structuration/Groupement__ : Clustering (basé sur features), Détection de Communautés (basée sur structure).   
__Validation/Exploration des Groupes__ : Analyse de modularité.   
__Applications/Analyses Spécifiques__ : Recommandations, Similarité, Pathfinding, autres Centralités.   
__Techniques d'Embedding Alternatives/Avancées__ : GraphSAGE.   

#### __`6.6 Préparation des variables et de l'environnement Neo4j`__

<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=500 large=200/>
</p>

#### __`6.7 Création du Graph projeté principal`__

In [ ]:
# Construction Dynamique des Propriétés
suffix = '_main'
graphName = 'graphName'

properties_Event=['eventTypeId','topicId']
properties_User=['friends_count', 'tweets_count','listed_count', 'statuses_count','favourites_count', 'followers_count', 'id']
properties_Tweet=['retweet_count','annotation_num_judgements','favorite_count','id','topicId']
properties_Hashtag=['occurences']


properties = [properties_Event, properties_User, properties_Tweet, properties_Hashtag, properties_PostCategory]
orientation= ['UNDIRECTED','UNDIRECTED','UNDIRECTED','UNDIRECTED','UNDIRECTED','UNDIRECTED','UNDIRECTED','UNDIRECTED','UNDIRECTED','UNDIRECTED']
nodeLabels = ['Event', 'Tweet', 'User', 'Hashtag', 'PostCategory']

gds=get_gdsConnection()
drop_wholeGraph(gds)

out=gds.run_cypher(query_create_main_graph, params={'graphName': 'graphName', 'properties':properties, 'orientation':orientation})

relationshipTypes_whole = ['IS_ABOUT','TALKS_ABOUT', 'MENTIONS', 'POSTED', 'REPLY_TO', 'RETWEETS', 'REPLIED_TO']
main_properties = [f'intermediateCommunities{suffix}', f'localClusteringCoefficient{suffix}']
dynamic_properties = [f'pageRank{suffix}', f'articulationPoint{suffix}', f'community{suffix}' ]
properties_toExtract = dynamic_properties + main_properties
all_properties = dynamic_properties + main_properties + properties

#### __`6.8 Application des algorithmes et persistance des variables sur Neo4j`__

__`Orientation : UNDIRECTED`__ 
 - `Leiden` community detection __Undirected graph heteregeneous nodes__ 
 - `localClusteringCoefficient` __Undirected graph, Heterogenous nodes__
 - `ArticulationPoint` __Undirected graph, Heterogenous nodes__
 - `labelPropagation` community detection __Undirected graph, Heterogenous nodes__ 

__`Orientation : REVERSE`__ 
 - `Eigenvector` vecteurs propres __Directed & Undirected graph, Heterogenous nodes__
 - `Betweenness` centralités intermédiaires __Directed & Undirected graph, Heterogenous nodes__
 - `Degrés` __Directed & Undirected graph, Heterogenous nodes__

In [ ]:
# pageRank 
clean_projectionGraph_Parameter(gds, graphName, f'pageRank{suffix}')
pageRank_df = gds.run_cypher(query_pageRank_mutate, params={'graphName': graphName, 
                                            'nodeProperties': f'pageRank{suffix}', 
                                            'nodeLabels': nodeLabels, 
                                            'relationshipTypes': relationshipTypes_whole })
# localClstrCoef 
clean_projectionGraph_Parameter(gds, graphName, f'localClusteringCoefficient{suffix}')
localClstrCoef_df = gds.run_cypher(localClstrCoef_query_mutate, params={'graphName': graphName,
                                            'localClusteringCoefficient': f'localClusteringCoefficient{suffix}'})  
# articulationPoints  
clean_projectionGraph_Parameter(gds, graphName, f'articulationPoint{suffix}')
articulationPoints_df = gds.run_cypher(articulationPoints_query_mutate, params={'graphName': graphName, 
                                            'articulationPoint': f'articulationPoint{suffix}'})
# labelPropagation  
clean_projectionGraph_Parameter(gds, graphName, f'community{suffix}')
labelPropagation_df = gds.run_cypher(query_labelPropagation_mutate, params={'graphName': graphName,
                                            'community': f'community{suffix}'})
# intermediateCommunities 
clean_projectionGraph_Parameter(gds, graphName, f'intermediateCommunities{suffix}')
intermediateCommunities_df = gds.run_cypher(query_leiden_mutate, params={'graphName': graphName, 
                                            'intermediateCommunities': f'intermediateCommunities{suffix}'}) 

# Créer le fastrp_embedding property sur le graphe projeté
clean_projectionGraph_Parameter(gds, graphName, 'fastrp_embedding')
fastrp_embedding_df = gds.run_cypher(query_mutate_embeddings,params={'graphName': graphName, 
                                            'mutateProperty': 'fastrp_embedding', 
                                            'nodeLabels': nodeLabels, 
                                            'relationshipTypes': relationshipTypes_whole})

# Créer le dregree property sur le graphe projeté
clean_projectionGraph_Parameter(gds, graphName, f'dregree{suffix}' )
degree_query = gds.run_cypher(degree_query_mutate, params={'graphName': graphName, 'degree': f'dregree{suffix}'})

# Créer le betweenness property sur le graphe projeté
clean_projectionGraph_Parameter(gds, graphName, f'betweeness{suffix}' )
betweenness_query = gds.run_cypher(betweenness_query_mutate, params={'graphName': graphName, 'betweenness':f'betweeness{suffix}' }) 

# Créer le eigenvector property sur le graphe projeté
clean_projectionGraph_Parameter(gds, graphName, f'eigenvector{suffix}' )
eigenvector_query = gds.run_cypher(eigenvector_query_mutate, params={'graphName': graphName, 'eigenvector':f'eigenvector{suffix}' }) 

In [ ]:
nodeProperties = [ f'pageRank{suffix}', 
                   f'localClusteringCoefficient{suffix}',f'articulationPoint{suffix}', 
                   f'community{suffix}', 
                   f'intermediateCommunities{suffix}', 
                   f'fastrp_embedding',
                   f'dregree{suffix}',
                   f'betweeness{suffix}',
                   f'eigenvector{suffix}']

node_df_prop = gds.run_cypher(query_nodeProperties, params={'graphName': graphName, 'nodeLabels': nodeLabels, 'nodeProperties': nodeProperties})
df_pivot = node_df_prop.pivot(index='originalId', columns=['nodeProperty'], values='propertyValue')
df_pivot = df_pivot.reset_index()
df_pivot

In [ ]:
relationship_details_df = extract_all_relationship_properties(gds, graphName)
relationship_details_df

In [ ]:
relationship_details_df.columns

In [ ]:
# Renommer les colonnes de df_pivot AVANT de fusionner
df_pivot_for_source = df_pivot.rename(columns={
    f'community{suffix}': 'source_community',
    f'fastrp_embedding': 'source_fastrp_embedding',
    f'localClusteringCoefficient{suffix}': 'source_localClusteringCoefficient',
    f'articulationPoint{suffix}': 'source_articulationPoint',
    f'intermediateCommunities{suffix}': 'source_intermediateCommunities',
    f'pageRank{suffix}': 'source_pageRank',
    f'dregree{suffix}':'source_dregree',
    f'betweeness{suffix}': 'source_betweeness',
    f'eigenvector{suffix}': 'source_eigenvector',
    'originalId': 'source_originalId'  
})

df_pivot_for_target = df_pivot.rename(columns={
    f'community{suffix}': 'target_community',
    f'fastrp_embedding': 'target_fastrp_embedding',
    f'localClusteringCoefficient{suffix}': 'target_localClusteringCoefficient',
    f'articulationPoint{suffix}': 'target_articulationPoint',
    f'intermediateCommunities{suffix}': 'target_intermediateCommunities',
    f'pageRank{suffix}': 'target_pageRank',
    f'dregree{suffix}':'target_dregree',
    f'betweeness{suffix}': 'target_betweeness',
    f'eigenvector{suffix}': 'target_eigenvector',
    'originalId': 'target_originalId' 
})  

In [ ]:

final_reconciled_df_source = pd.merge(
    relationship_details_df,
    df_pivot_for_source,
    left_on='sourceElementId', 
    right_on='source_originalId',  
    how='left'
)
final_reconciled_df_source

In [ ]:
final_reconciled_df_target = pd.merge(
    relationship_details_df,
    df_pivot_for_target,
    left_on='targetElementId', 
    right_on='target_originalId',  
    how='left'
)
final_reconciled_df_target

In [ ]:

final_reconciled_df = pd.merge(
    final_reconciled_df_source,
    df_pivot_for_target,
    left_on='targetElementId',
    right_on='target_originalId',
    how='left',
    suffixes=('_source', '_target')
)
final_reconciled_df.drop(columns=['sourceGdsNode','targetGdsNode','sourceElementId','targetElementId'], inplace=True)
final_reconciled_df

__`Analyse en Colonne (Analyse de Métriques Individuelles)`__
   
`localClusteringCoefficient:` Effectue une mesure de la densité locale fournissant une valeur comprise entre 0 et 1

- __Valeur proche de 1__ : Indique que la plupart (ou tous) les voisins du nœud sont connectés entre eux. Le voisinage du nœud ressemble fortement à une clique (un groupe où tout le monde est connecté à tout le monde). Un nœud avec un LCC de 1.0 a tous ses voisins directement connectés les uns aux autres.

- __Valeur proche de 0__ : Indique que les voisins du nœud sont peu ou pas connectés entre eux. Le nœud peut agir comme un "pont" entre différents groupes ou individus qui ne se connaissent pas directement. Un LCC de 0.0 signifie qu'aucun des voisins du nœud n'est connecté à un autre voisin.    

In [ ]:
print(f"Répertorie le proprata des pomesure de dentsité local pour les points d'articulation ")
final_reconciled_df['source_localClusteringCoefficient'].value_counts().head()

`L'algorithme PageRank` permet de mesurer la pertinence de chaque nœud d'un graphe, qui dépend du nombre de relations entrantes provenant des autres nœuds et de l'importance des nœuds sources. 

 - Objectif : Calculer l'influence des nœuds.
 - Analyse : Très utile pour obtenir une première mesure de l'importance des nœuds. C'est une étape pour identifier les acteurs clés.

In [ ]:
print(f"Compter source_pageRank ")
final_reconciled_df['source_pageRank'].value_counts().head()


`intermediateCommunities:` Fournit une liste d'entiers qui encode les différentes niveaux hiérarchiques de communautés auxquelles un nœud appartient. 

Les communautés détectées représentent des groupes de nœuds densément connectés entre eux mais faiblement connectés aux autres groupes. Ces structures sont représentatives de sous-ensembles significatifs des données, comme :

 - Des groupes d'utilisateurs ayant des interactions fréquentes.
 - Des clusters thématiques (hashtags ou sujets similaires).
 - Des événements ou des discussions qui sont fortement liés entre eux.

In [ ]:
final_reconciled_df['source_intermediateCommunities'].value_counts()

In [ ]:

print(f"Analyser la distribution des tailles des communautés trouvées par Label Propagation")
final_reconciled_df['target_intermediateCommunities'].value_counts()

In [ ]:
print(f"Analyser la distribution des tailles des communautés trouvées par coefficient local de clustering")
final_reconciled_df['target_localClusteringCoefficient'].value_counts()

__`Analyse en Ligne (Analyse de Métriques Individuelles)`__  

 - Objectif : Comprendre le profil complet d'un nœud spécifique en regardant toutes ses métriques, ou comparer les profils de plusieurs nœuds.    
 - Comment : Se concentrer sur une ou plusieurs lignes à la fois, en considérant plusieurs colonnes simultanément.

In [ ]:
import pandas as pd

def nodes_compare(final_reconciled_df, node_1, node_2):
    # Mettre nodeId comme index pour faciliter la sélection par ligne
    df_metrics_indexed = final_reconciled_df.set_index('source_originalId')
    
    # Profil du nœud 1
    # .loc[node_1] peut retourner un DataFrame si node_1 est un index dupliqué
    node_1_profile_df = df_metrics_indexed.loc[node_1]
    # S'assurer que c'est un DataFrame même si une seule ligne est retournée, pour être cohérent
    if isinstance(node_1_profile_df, pd.Series):
        node_1_profile_df = node_1_profile_df.to_frame().T
        node_1_profile_df.index.name = 'sourceOriginalId' # Restaurer le nom de l'index s'il est perdu
        
    #print(f"1 Profil du Nœud {node_1}:\n{node_1_profile_df}\n")

    # Profil du nœud 2
    node_2_profile_df = df_metrics_indexed.loc[node_2]
    if isinstance(node_2_profile_df, pd.Series):
        node_2_profile_df = node_2_profile_df.to_frame().T
        node_2_profile_df.index.name = 'sourceOriginalId'

    #print(f"2 Profil du Nœud {node_2}:\n{node_2_profile_df}\n")
    
    node_1_articulation_point = node_1_profile_df['source_articulationPoint'].iloc[0]
    # On s'attend à ce que 'source_intermediateCommunities' soit une liste
    node_1_communities = node_1_profile_df['source_intermediateCommunities'].iloc[0]

    node_2_articulation_point = node_2_profile_df['source_articulationPoint'].iloc[0]
    node_2_communities = node_2_profile_df['source_intermediateCommunities'].iloc[0]

    # Comparaison simple
    if node_2_articulation_point == 1:
        print(f"1 Le Nœud {node_2} est un point d'articulation.")
    else:
        print(f"1 Le Nœud {node_2} n'est PAS un point d'articulation.")
        
    if node_1_articulation_point == 1:
        print(f"2 Le Nœud {node_1} est un point d'articulation.")
    else:
        print(f"2 Le Nœud {node_1} n'est PAS un point d'articulation.")

    # Comparaison de listes pour l'égalité
    if node_1_communities == node_2_communities:
        print("Les deux nœuds sont dans la même communauté selon Label Propagation.")
    else:
        print("Les deux nœuds sont dans des communautés différentes selon Label Propagation.")


node_1 = '4:4da88401-a5f5-4fc8-9db2-077d56e88509:0'
node_2 = '4:4da88401-a5f5-4fc8-9db2-077d56e88509:1'
nodes_compare(final_reconciled_df, node_1, node_2)


In [ ]:
# Effectue une réduction afin de permettre l'affichage
gds.run_cypher( query_random_walk, params={
                'subgraph_name': 'graphName',
                'sampled_graph_name': 'sampled_mainGraph' ,
                'sampling_ratio':0.01,
                'nodeProperties': all_properties
                })
#out=gds.run_cypher(query_drop, params={'graphName': 'mainGraph'})
list_graphs(gds)

In [ ]:
# Rendu avec paramètres de performance
VG = display_intermediate(gds,'sampled_mainGraph',f'pageRank{suffix}')
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

#### __`7.1 Interprétaton des résultats de Coefficient de clustering local`__

`Le coefficient de clustering local évalue la densité de connexion locale entre les voisins immédiats d'un nœud`

__`Interprétation`__

 - Valeur élevée (proche de 1) : Indique une communauté très soudée (ex. famille dans un réseau social)
 - Valeur faible (proche de 0) : Réseau local peu dense ou structure en étoile

__`Particularités techniques : `__ 
 - Calcul orienté/non orienté : Certains algorithmes ignorent la direction des arêtes pour simplifier le calcul.

__`Cette mesure révèle des micro-structures cachées dans les réseaux complexes, avec des implications en sociologie, épidémiologie et science des données`__

#### __`7.1.1 Interprétaton des résultats de visualisation du Coefficient de clustering local`__ 

L’image du graphe complet montre une composante géante très dense et de nombreux petits amas ou nœuds isolés.       
Cela confirme la structure “hub-and-spoke” : beaucoup de périphérie peu connectée, quelques centres denses.

__`En résumé`__

Ce graphe est typique d’un réseau social large, avec une immense majorité de nœuds faiblement connectés localement (LCC=0), et quelques petits groupes très denses (LCC=1). 

Les Tweets, Users et Hashtags partagent ce profil, mais les Users ont un peu plus de diversité dans leurs valeurs de clustering.     
La structure globale est très éclatée, ce qui limite la propagation locale mais favorise la diffusion rapide via des hubs.

In [ ]:
# Rendu avec paramètres de performance
VG = display_intermediate(gds,'sampled_mainGraph',f'localClusteringCoefficient{suffix}')
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

 :::

 __`Cette visualisation révèle une topologie de "petit monde" (small-world network) avec la possibilité :`__
 
   
 - __Identifier les communautés__ : Utiliser le coefficient de clustering pour repérer les groupes d'utilisateurs ou d'événements fortement interconnectés
 - __Analyser la résilience du réseau__ : Évaluer comment la suppression de certains nœuds affecte la connectivité globale
 - __Détecter des anomalies__ : Identifier des nœuds avec un coefficient de clustering anormalement bas ou élevé par rapport à leurs voisins
 - __Évaluer la diffusion d'information__ : Analyser comment l'information se propage à travers le réseau en fonction de la structure locale
 - __Identifier les influenceurs__ : Repérer les nœuds avec un coefficient de clustering élevé qui pourraient jouer un rôle clé dans la diffusion d'information
 
 ::: 

L'image montre un réseau avec deux structures principales en forme d'étoile et plusieurs nœuds colorés (violet, orange, "orangé clair", jaune, bleu) qui se distinguent par leur taille et leur coloration, indiquant probablement des coefficients de clustering local élevés. 

 ::: 

__`Analyses complémentaires recommandées`__
 - Croiser ces informations avec d'autres métriques (centralité de proximité, betweenness)
 - Comparer les coefficients moyens entre différentes communautés
 - Établir une distribution statistique des coefficients pour identifier des seuils significatifs    

 :::   

In [ ]:
# Rendu avec paramètres de performance
VG = display_intermediate(gds,'sampled_mainGraph',f'dregree{suffix}')
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

In [ ]:
# Rendu avec paramètres de performance
VG = display_intermediate(gds,'sampled_mainGraph',f'betweeness{suffix}')
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

In [ ]:
# Rendu avec paramètres de performance
VG = display_intermediate(gds,'sampled_mainGraph',f'eigenvector{suffix}')
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

### __`7.2 Interprétation des résultats de l'algorithme Point d'articulation`__

::: 

__`Un point d'articulation est un nœud dont la suppression augmente le nombre de composants connectés dans le graphe.`__

__`Implications pratiques`__  

__Vulnérabilité structurelle__ : Ces points d'articulation constituent des points de défaillance uniques dont la suppression fragmenterait davantage le réseau
__Contrôle informationnel__ : Ces nœuds exercent un contrôle disproportionné sur les flux d'information entre différentes parties du réseau     
__Cibles d'intervention__ : Pour une intervention stratégique, ces nœuds offriraient le meilleur rapport impact/effort.
 
 :::

#### __`7.2.1 Interprétation des résultats de visualisation des points d'articulation`__

:::

 - Les points d'articulation sont mis en évidence par des nœuds bleus de plus grande taille.     
 - Ces nœuds particuliers révèlent des informations critiques sur la structure du graphe.

__`Structure globale du réseau`__

 - Le graphe est extrêmement fragmenté, composé de multiples composantes connexes isolées.    
 - La distribution spatiale montre une faible densité globale avec de nombreux espaces vides   
 - En bleu les points d'articulation dispersés dans le réseau, indiquent des connexions critiques entre différentes parties du graphe.

__`Implications pour l'analyse du réseau`__
 - __Faible connectivité globale__ : Le réseau est peu dense, avec de nombreux nœuds isolés.    
 - __Points d'articulation critiques__ : Ces nœuds sont essentiels pour maintenir la connectivité du réseau.    
 - __Vulnérabilité structurelle__ : La suppression de ces points d'articulation entraînerait une fragmentation supplémentaire du réseau. 
 - __Faible redondance des chemins__ : L'absence de circuits alternatifs rend la communication entre nœuds très dépendante de ces points critiques.
 - __Centralisation locale__ : Certaines zones montrent une organisation centralisée autour d'un point d'articulation unique.


:::

::: 
__`Structure Globale__ : L'image montre une visualisation de votre graphe. On observe une composante principale très large et dense, ainsi que plusieurs petits groupes de nœuds isolés et des nœuds complètement isolés. C'est une structure assez typique pour les réseaux sociaux ou les graphes d'interaction. 

Cette visualisation du réseau présente des caractéristiques révélatrices issues des points d'articulation du graphe :

__`Structure et implications`__
- Les nœuds de couleur rouge représentent des points d'articulation, indiquant des connexions critiques entre différentes parties du réseau.
- Les nœuds de couleur bleu  sont des nœuds isolés, suggérant qu'ils ne sont pas directement connectés aux points d'articulation.

:::

:::

__`Observations clés`__  

Signification du  Modèle : Si l'algorithme identifie un nœud comme point d'articulation :

__`Si c'est un User`__ : Cet utilisateur est crucial pour connecter différentes parties de la communauté ou différentes conversations. Sa suppression "couperait" le graphe en plusieurs morceaux non connectés (en termes de chemin, ignorant l'orientation initiale des liens).    

__`Si c'est un Tweet`__ : Ce tweet spécifique sert de pont essentiel entre différentes discussions ou groupes d'utilisateurs qui ne seraient sinon pas reliés (toujours dans une vue non orientée du graphe).    

__`Si c'est un Hashtag ou Event`__ : Ce hashtag ou cet événement est un connecteur unique entre des groupes d'utilisateurs ou des conversations qui, sans lui, seraient séparés.

Comment les Identifier : L'algorithme GDS écrit généralement une propriété (par exemple, articulationPoint) sur les nœuds. Ceux ayant la valeur 1 sont des points d'articulation, les autres (avec 0) ne le sont pas. Il faudrait interroger la base de données pour trouver ces nœuds spécifiques après avoir exécuté l'algorithme en mode mutate ou write.

:::

:::

__Le schéma suggère un réseau peu résilient avec des chemins de communication vulnérables. Dans un contexte d'analyse d'événements, ces points d'articulation pourraient représenter des événements charnières reliant différentes catégories ou phases d'incidents.__

:::


In [ ]:
# Rendu avec paramètres de performance
VG = display_intermediate(gds,'sampled_mainGraph',f'articulationPoint{suffix}')
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

### __`7.3 Interprétation des résultats de l'algorithme LPA labelPropagation `__

 - __`labelPropagation LPA est un algorithme rapide permettant de trouver des communautés dans un graphe.`__          
 - __`Il détecte ces communautés en se basant uniquement sur la structure du réseau.`__

##### __`7.3.1 Interprétaton des résultats de visualisation de l'algorithme LPA labelPropagation`__

__`Structure générale:`__
- Le graphe est composé de plusieurs clusters (groupes de nœuds), certains plus denses que d'autres.      
- Les nœuds jaunes représentent des nœuds centraux, ayant atteint un consensus sur leur label après la propagation.      
- Les zones plus espacées ou isolées indiquent des communautés moins connectées ou des clusters périphériques.      

__`Communautés détectées:`__

 - Chaque groupe dense correspond à une communauté où les nœuds partagent le même label.
 - Les connexions entre clusters sont peu nombreuses, ce qui suggère une faible interconnexion entre communautés.

__`Points critiques:`__

 - Les nœuds jaunes de grande taille peuvent être interprétés comme des pivots ou des influenceurs au sein de leurs communautés respectives.    

__`7.3.2. Analyse des communautés`__

__Identification des clusters :__ Chaque groupe dense peut représenter une communauté fonctionnelle ou thématique dans le réseau.      
__Frontières communautaires :__ Les zones où les labels ne se propagent (espaces entre clusters) indique des barrières structurelles ou des sous-groupes isolés.

__`7.3.3. Ciblage stratégique`__

Les nœuds jaunes (centraux) peuvent être utilisés pour :

 - Diffuser rapidement de l'information au sein d'une communauté.     
 - Relier différentes communautés pour favoriser une communication inter-groupes.

__`7.3.4. Optimisation du réseau`__

 - Réduire les barrières entre clusters en ajoutant des connexions stratégiques.      
 - Identifier les nœuds isolés pour les intégrer dans des communautés existantes.

__`7.3.5. Applications possibles`__  

- __Analyse de réseaux sociaux:__ Identifier des groupes d'utilisateurs partageant des intérêts communs.    
- __Analyse de réseaux de transport:__ Identifier des sous-réseaux de transport interconnectés.   
- __Analyse de réseaux de communication :__ Identifier des groupes d'utilisateurs partageant des intérêts communs. 
- __Analyse de réseaux de citation:__ Identifier des groupes d'auteurs ou de publications interconnectés.
- __Analyse de réseaux de co-auteurs:__ Identifier des groupes d'auteurs ou de publications interconnectés.   
    
__`7.3.6. Limites et recommandations`__

 - __`Absence de labels explicites :`__ Colorés par label, les noeuds sont sans étiquettes explicites, il est difficile d'en comprendre la signification.
 - __`Non-unicité des solutions :`__ L'algorithme LPA peut produire différentes solutions selon l'ordre initial des nœuds ou les paramètres choisis.
 - __`Densité variable :`__ Les groupes denses sont identifiables, mais les zones moins connectées nécessitent une analyse supplémentaire.

In [ ]:
# Rendu avec paramètres de performance
gds=get_gdsConnection()
VG = display_intermediate(gds,'sampled_mainGraph','community_main')
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

In [ ]:
list_graphs(gds)

#### __`7.5 Interprétation des résultats de l'algorithme Leiden de détection des communautés`__

L'algorithme de Leiden est un algorithme de détection de communautés dans les grands réseaux.    
Il sépare les nœuds en communautés disjointes afin de maximiser le score de modularité de chaque communauté.     
La modularité quantifie la qualité de l'affectation des nœuds aux communautés, c'est-à-dire la densité de connexion des nœuds d'une communauté par rapport à leur niveau de connectivité dans un réseau aléatoire.

__`7.5.1 Visualisation de l'algorithme Leiden`__

 - __Le graphe est composé de plusieurs clusters (groupes de nœuds), certains plus denses que d'autres.__
 - __Les nœuds jaunes représentent des nœuds centraux, ayant atteint un consensus sur leur label après la propagation.__
 - __Les zones plus espacées ou isolées indiquent des communautés moins connectées ou des clusters périphériques.__
 - __Chaque groupe dense correspond à une communauté où les nœuds partagent le même label.__
 - __Les connexions entre clusters sont peu nombreuses, ce qui suggère une faible interconnexion entre communautés.__
 - __Les nœuds jaunes de grande taille peuvent être interprétés comme des pivots ou des influenceurs au sein de leurs communautés respectives.__
 - __Les nœuds de couleur orange et jaune sont des nœuds isolés, suggérant qu'ils ne sont pas directement connectés aux points d'articulation.__    

In [ ]:
# Liste les composantes du sous graphe projeté définit
gds.run_cypher("""CALL gds.graph.list('sampled_mainGraph') YIELD schema RETURN schema""")

In [ ]:
def color_communities_by_level(nodes, level, cmap_name):
    # Extraire les IDs de communauté pour le niveau spécifié
    community_ids = nodes['propertyValue'].apply(lambda x: x[level-1])
    
    # Générer une palette discrète
    unique_comms = community_ids.unique()
    cmap = plt.get_cmap(cmap_name, len(unique_comms))
    color_mapping = {comm: matplotlib.colors.rgb2hex(cmap(i)) for i, comm in enumerate(unique_comms)}
    
    return [color_mapping[comm] for comm in community_ids]

In [ ]:
graph_obj = gds.graph.get("sampled_mainGraph")
nodes = gds.graph.nodeProperties.stream(graph_obj,['intermediateCommunities_main'])
nodes

__`Les listes propertyValue représente une hiérarchie de communautés issue de l'algorithme Leiden.`__ 

`Voici comment l'interpréter propertyValue [75944, 780, 57, 70, 1752] :`

 - __Niveau 1 (le plus fin) :__ Le nœud appartient à une petite communauté locale dont l'ID est 75944, trouvée lors de la première phase de l'algorithme.      
 - __Niveau 2 :__ La communauté 75944 a été regroupée dans une communauté plus large de niveau 2, dont l'ID est 780.      
 - __Niveau 3 :__ La communauté 780 est elle-même regroupée dans une communauté encore plus grande de niveau 3, avec l'ID 57.      
 - __Niveau 4 :__ La communauté 57 fait partie de la communauté 70.      
 - __Niveau 5 (le plus grossier) :__ La communauté 70 fait partie de la communauté 1752.      

__`En résumé :`__

    Le nœud appartient directement à une seule communauté à chaque niveau de la hiérarchie. La liste montre la séquence d'appartenance à des communautés de plus en plus larges au fur et à mesure que l'algorithme agrège les groupes. Ce n'est pas un cas de communautés recouvrantes où un nœud a plusieurs appartenances distinctes et simultanées au même niveau. 

__C'est une structure imbriquée :__ la communauté de niveau 1 est contenue dans celle de niveau 2, qui est contenue dans celle de niveau 3, et ainsi de suite. - 

In [ ]:
# Leiden 
import matplotlib.pyplot as plt
import matplotlib.colors
from neo4j_viz.gds import from_gds


def display_intermediate(gds, graph_obj, niveau_a_visualiser, colormap_base):
    # Créer une fonction pour colorer les communautés par niveau
    nodes = gds.graph.nodeProperties.stream(graph_obj,['intermediateCommunities_main'])

    node_colors_for_level = color_communities_by_level(nodes, niveau_a_visualiser, cmap_name=colormap_base)
    node_id_to_color_map = pd.Series(node_colors_for_level, index=nodes['nodeId'] ).to_dict()

    VG = from_gds(
            gds,
            graph_obj,
            size_property="pageRank_main"
        )

    # Appliquer pour chaque niveau
    VG.color_nodes('id', node_id_to_color_map)

    return VG

niveau_a_visualiser = 5
colormap_base = 'viridis'
graph_obj = gds.graph.get("sampled_mainGraph")
gds=get_gdsConnection()

VG=display_intermediate(gds, graph_obj, niveau_a_visualiser, colormap_base)
# Rendu avec paramètres de performance
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

In [ ]:
# viridis' # Ou 'plasma', 'inferno', 'tab20', 'tab20b'
niveau_a_visualiser = 4
colormap_base = 'plasma'
graph_obj = gds.graph.get("sampled_mainGraph")
gds=get_gdsConnection()

VG= display_intermediate(gds, graph_obj, niveau_a_visualiser, colormap_base)
# Rendu avec paramètres de performance
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

In [ ]:
# viridis' # Ou 'plasma', 'inferno', 'tab20', 'tab20b'
niveau_a_visualiser = 3
colormap_base = 'inferno'
graph_obj = gds.graph.get("sampled_mainGraph")
gds=get_gdsConnection()

VG= display_intermediate(gds, graph_obj, niveau_a_visualiser, colormap_base)
# Rendu avec paramètres de performance
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

In [ ]:
# viridis' # Ou 'plasma', 'inferno', 'tab20', 'tab20b'
niveau_a_visualiser = 2
colormap_base = 'tab20'
graph_obj = gds.graph.get("sampled_mainGraph")
gds=get_gdsConnection()

VG= display_intermediate(gds, graph_obj, niveau_a_visualiser, colormap_base)
# Rendu avec paramètres de performance
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

In [ ]:
# viridis' # Ou 'plasma', 'inferno', 'tab20', 'tab20b'
niveau_a_visualiser = 1
colormap_base = 'tab20b'
graph_obj = gds.graph.get("sampled_mainGraph")
gds=get_gdsConnection()

VG= display_intermediate(gds, graph_obj, niveau_a_visualiser, colormap_base)
# Rendu avec paramètres de performance
VG.render( width="100%", height="600px", max_allowed_nodes=11000)

#### __`7.4 Interprétation des Résultats`__

Les calculs effectués comme le PageRank, Leiden, articulation points reposent sur des métriques structurelles du graphe. 

Ces algorithmes exploitent directement, les nœuds, les relations les propriétés scalaires associées aux nœuds (pageRank_main ou community_main).    
__Ils permettent de comprendre la structure du graphe et d'identifier des clusters ou des points critiques sans avoir avoir à transformer les nœuds en vecteurs via un embedding__.

`community_main:` LPA labelPropagation détecte les communautés en se basant uniquement sur la structure du réseau.     
`localClusteringCoefficient:` Effectue une mesure de la densité locale.     
`intermediateCommunities:` Fournit une liste d'entiers qui encode les différentes niveaux hiérarchiques de communautés auxquelles un nœud appartient. 

Les communautés détectées représentent des groupes de nœuds densément connectés entre eux mais faiblement connectés aux autres groupes. Ces structures sont représentatives de sous-ensembles significatifs des données, comme :

 - Des groupes d'utilisateurs ayant des interactions fréquentes.
 - Des clusters thématiques (hashtags ou sujets similaires).

Les cercles représentent des communautés ou clusters détectés par l'algorithme Leiden, labelPropagation ou Label Propagation.
 - Les couleurs appliquées permettent de différencier les communautés et/ou les points d'articulation critiques.
 - Les tailles des nœuds reflètent leur importance selon le score PageRank.

Les cercles ou clusters visibles peuvent être interprétés comme suit :

 - __Grand cercle__ : Une communauté dominante ou un nœud central avec un PageRank très élevé.
 - __Petits cercles connectés__ : Sous-communautés ou clusters secondaires connectés au cercle principal.
 - __Clusters isolés__ : Des groupes indépendants qui ne sont pas fortement connectés au reste du graphe



#### __`7.8 Libère la mémoire des grahpes projetés`__ 

In [ ]:
# Libère la mémoire des sous-graphes stockés
#drop_wholeGraph(gds) 

#### __`7.9 Retire les attributs et efface les données ajoutées à la base Neo4j`__ 

In [ ]:
# Nettoie la base de donnée Neo4j des paramètres qui ont été ajoutés
#clean_database_parameter(gds)

#### __`7.8 Liste les attributs des différents labels en base Neo4j`__ 

In [ ]:
# Liste les attributs des différents labels de la base Neo4j
query_listLabelAttributs="""
    MATCH (n) 
    RETURN DISTINCT labels(n), keys(n)
    """
gds=get_gdsConnection()
gds.run_cypher(query_listLabelAttributs)

In [ ]:
query_main_graph = """
CALL gds.graph.project(
    $graphName,
    {
        Event: {label: 'Event', properties: { eventTypeId: { property: 'eventTypeId' } }},
        User: { label: 'User' },        
        Tweet: { label: 'Tweet' }, 
        Hashtag: { label: 'Hashtag' },
        PostCategory: { label: 'PostCategory' }
    },
    ['IS_ABOUT', 'TALKS_ABOUT', 'MENTIONS', 'POSTED', 'REPLY_TO', 'RETWEETS', 'REPLIED_TO']
)
YIELD graphName, nodeCount, relationshipCount
"""
gds=get_gdsConnection()
gds.graph.drop('graphName')
out=gds.run_cypher(query_main_graph, params={'graphName': 'graphName'})

In [ ]:
query_CountTweetByEventId = """
MATCH (e:Event)
WITH e.eventType AS Event, 
     count(e.eventTypeId) AS EventCount, 
     collect(DISTINCT e.trecisid) AS TrecisIds, e.eventTypeId as EventId 
MATCH (e:Event)<-[r:IS_ABOUT]-(t:Tweet)
WHERE t.topic IN TrecisIds 
RETURN EventId, Event, EventCount, count(r) AS TweetCount
"""
gds.run_cypher(query_CountTweetByEventId)

In [ ]:
gds.run_cypher(query_listEventType)

In [ ]:
query_create_main_graph = """
    CALL gds.graph.project(
        $graphName,
        {
          Event:        { properties: $properties_Event },
          User:         { properties: $properties_User },
          Tweet:        { properties: $properties_Tweet },
          Hashtag:      { properties: $properties_Hashtag },
          PostCategory: { properties: $properties_PostCategory }
        },
        {
          POSTED:      { orientation: $orientation },
          MENTIONS:    { orientation: $orientation },
          REPLIED_TO:  { orientation: $orientation },
          TALKS_ABOUT: { orientation: $orientation },
          REPLY_TO:    { orientation: $orientation },
          IS_ABOUT:    { orientation: $orientation },
          HAS_HASHTAG: { orientation: $orientation },
          HAS_CATEGORY:{ orientation: $orientation },
          RETWEETS:    { orientation: $orientation }
        }   
    )
    YIELD graphName, nodeCount, relationshipCount; 
"""
properties_Event=['eventTypeId']
properties_User=[]
properties_Tweet=[]
properties_Hashtag=[]
properties_PostCategory=[] 
#orientation='UNDIRECTED'
#gds=get_gdsConnection()
#gds.graph.drop('graphName')
#out=gds.run_cypher(query_create_main_graph, params={'graphName': 'graphName', 'properties':properties, 'orientation':orientation})

In [ ]:
# First, retrieve the trecisid_list for the given event_id
event_id = 5
trecisid_query = """
MATCH (e:Event {eventTypeId: $event_id})
RETURN collect(e.trecisid) AS trecisid_list
"""
result = gds.run_cypher(trecisid_query, params={'event_id': event_id})
trecisid_list = result.iloc[0]['trecisid_list']

# Now, use the retrieved trecisid_list in the graph projection query
query_filter_subgraph = """
CALL gds.graph.project.cypher(
    $filtered_graph,
    'MATCH (n) 
     WHERE (n:Event AND n.eventTypeId = $event_id) OR 
           (n:Tweet AND n.topic IN $trecisid_list) 
     RETURN id(n) AS id, labels(n) AS labels',
     'MATCH (n)-[r]->(m) 
     WHERE (n:Event AND n.eventTypeId = $event_id) OR 
           (m:Event AND m.eventTypeId = $event_id) 
     RETURN id(n) AS source, 
            id(m) AS target, 
            type(r) AS type',
    { parameters: { event_id: $event_id, trecisid_list: $trecisid_list }, validateRelationships: false }
)
YIELD  graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
"""
gds.graph.drop('filtered_graph_5')
gds.run_cypher(query_filter_subgraph, params={'filtered_graph': 'filtered_graph_5', 'event_id': event_id, 'trecisid_list': trecisid_list})


In [ ]:
properties=['eventTypeId']
orientation='UNDIRECTED'

drop_wholeGraph(gds) 
out=gds.run_cypher(query_create_main_graph,params={'graphName': 'mainGraph', 'properties': properties, 'orientation': orientation})

#### __`7.6 Liste les différents sous-graphes existants en mémoire`__ 

In [ ]:
# Liste les différents sous-graphe existant en mémoire
gds.run_cypher("""CALL gds.graph.list() YIELD graphName,nodeCount,relationshipCount, schema RETURN distinct graphName, nodeCount,relationshipCount""")

#### __`7.7 Liste les composantes du sous graphe projeté`__ 

In [ ]:
# Liste les composantes du sous graphe projeté définit
gds.run_cypher("""CALL gds.graph.list('mainGraph') YIELD schema RETURN schema""")

### __`8 Sous graphe projeté par type d'Event`__  

In [ ]:
query_filter_subgraph = """
CALL gds.graph.filter(
    $filtered_graph,
    $mainGraph,
    '(n:Event AND n.eventTypeId = $event_id)',
    'r:IS_ABOUT OR r:TALKS_ABOUT OR r:MENTIONS OR r:POSTED OR r:REPLY_TO OR r:RETWEETS OR r:REPLIED_TO',
     { parameters: { event_id: $event_id } }
)"""
gds=get_gdsConnection()
gds.graph.drop('filtered_graph_15')
gds.run_cypher(query_filter_subgraph, params={'mainGraph': 'mainGraph','filtered_graph': 'filtered_graph_15', 'event_id': 5})

In [ ]:
query_create_filtered_subgraph = """
CALL gds.graph.project.cypher(
    $graphName,
    // Projection des nœuds (version corrigée avec EXISTS explicite)
    '
    MATCH (n)
    WHERE 
        (n:Event AND n.eventTypeId = $event_id) 
        OR (n:Tweet AND EXISTS {
            MATCH (e:Event {eventTypeId: $event_id})
            WHERE n.topic = e.trecisid
        })
        OR (n:Hashtag AND EXISTS {
            MATCH (n)<-[:HAS_HASHTAG]-(t:Tweet)
            MATCH (e:Event {eventTypeId: $event_id})
            WHERE t.topic = e.trecisid
        })
        OR (n:PostCategory AND EXISTS {
            MATCH (n)<-[:HAS_CATEGORY]-(t:Tweet)
            MATCH (e:Event {eventTypeId: $event_id})
            WHERE t.topic = e.trecisid
        })
    RETURN id(n) AS id, labels(n) AS labels
    ',
    // Projection des relations (version simplifiée)
    '
    MATCH (t:Tweet)-[r:IS_ABOUT]->(e:Event {eventTypeId: $event_id})
    WHERE t.topic = e.trecisid
    WITH t, e
    MATCH (t)-[rel:POSTED|HAS_HASHTAG|HAS_CATEGORY|MENTIONS|REPLY_TO|RETWEETS]->(m)
    RETURN id(t) AS source, id(m) AS target, type(rel) AS type
    UNION
    MATCH (t:Tweet)-[r:IS_ABOUT]->(e:Event {eventTypeId: $event_id})
    WHERE t.topic = e.trecisid
    RETURN id(t) AS source, id(e) AS target, "IS_ABOUT" AS type
    ',
    {parameters: {event_id: $event_id}, validateRelationships: false}
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
"""

gds=get_gdsConnection()
gds.graph.drop('filtered_graph_1')
gds.run_cypher(query_create_filtered_subgraph, params={'graphName': 'filtered_graph_55', 'event_id': 5}) 

In [ ]:
query_create_filtered_subgraph = """
CALL gds.graph.project.cypher(
    $graphName,
    
    'MATCH (n)
     WHERE (n:Event AND n.eventTypeId = $event_id)
           OR (n:Hashtag)  // Include Hashtag nodes
           OR (n:PostCategory)  // Include PostCategory nodes
           OR EXISTS((n)<-[]-(:Event {eventTypeId: $event_id}))
           OR EXISTS((n)-[]->(:Event {eventTypeId: $event_id}))
     RETURN id(n) AS id, labels(n) AS labels',
    
    'MATCH (n)-[r]->(m)
     WHERE 
       (n:Event AND n.eventTypeId = $event_id) OR 
       (m:Event AND m.eventTypeId = $event_id) OR
       (EXISTS((n)<-[]-(:Event {eventTypeId: $event_id})) AND 
        EXISTS((m)-[]->(:Event {eventTypeId: $event_id})))
     RETURN id(n) AS source, id(m) AS target, type(r) AS type',
    
    {parameters: {event_id: $event_id}}
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
"""


#    MATCH (e:Event)<-[:IS_ABOUT]-(t:Tweet)
#     MATCH (e:Event)
#    MATCH (t:Tweet {topic: e.trecisid})  

gds=get_gdsConnection()
gds.graph.drop('filtered_graph_1')
gds.run_cypher(query_create_filtered_subgraph, params={'graphName': 'filtered_graph_25', 'event_id': 5}) 

In [ ]:
# Liste les différents sous-graphe existant en mémoire
gds.run_cypher("""CALL gds.graph.list() YIELD graphName,nodeCount,relationshipCount, schema RETURN distinct graphName, nodeCount,relationshipCount""")

In [ ]:
#gds.graph.drop('filtered_graph_name')

In [ ]:
gds.run_cypher("""CALL gds.graph.list('userInfluence') YIELD schema RETURN schema""")

__`1. Projection initiale (sans filtres)`__   
__`2. Filtrage par Event_ID`__    
__`3. Sampling CNARW (sur sous-graphe filtré)`__     
__`4. Calcul des Métriques :`__      
   - __`a) Label Propagation (détection communautés)`__     
   - __`b) Degree Centrality`__     
   - __`c) PageRank (pondéré par les communautés)`__    
   - __`d) Betweenness`__      
   - __`e) Eigenvector`__     
      
__`5. Écriture des propriétés`__     
__`6. Nettoyage des graphes temporaires`__   

In [ ]:
# Liste les différents sous-graphe existant en mémoire
gds.run_cypher("""CALL gds.graph.list() YIELD graphName, schema RETURN distinct graphName""")
# Liste les composantes du sous graphe projeté définit
gds.run_cypher("""CALL gds.graph.list('filtered_graph_name') YIELD schema RETURN schema""")
# Liste les attributs des différents labels de la base Neo4j
query_listLabelAttributs="""
    MATCH (n) 
    RETURN DISTINCT labels(n), keys(n)
    """
gds.get_gdsConnection()
gds.run_cypher(query_listLabelAttributs)
# Libère la mémoire des sous-graphes stockés
drop_wholeGraph(gds)
# Nettoie la base de donnée Neo4j des paramètres qui ont été ajoutés
clean_database_parameter(gds)
# Liste les différents sous-graphe existant en mémoire
gds.run_cypher("""CALL gds.graph.list() YIELD graphName, schema RETURN distinct graphName""")
# Liste les composantes du sous graphe projeté définit

In [ ]:
query_create_main_graph ="""
EXPLAIN
MATCH (e:Event {eventTypeId: $event_id})
WHERE e.trecisid IS NOT NULL
WITH e
MATCH (t:Tweet {topic: e.trecisid})
MATCH (u:User)-[:POSTED]->(t)
MATCH (t)-[:HAS_HASHTAG]->(h:Hashtag)
MATCH (t)-[:HAS_CATEGORY]->(pc:PostCategory)
RETURN u.name AS username, t.id AS tweet, t.retweet_count AS retweet_count,
       t.favorite_count AS favorite_count, h.id AS hash_val
ORDER BY t.favorite_count DESC, t.retweet_count DESC
LIMIT 10;
               
          """
gds.run_cypher(query_create_main_graph , params={'event_id': 1})

In [ ]:

#Ensuite filtrer pour un type d'événement spécifique
query_filter_subgraph = """
CALL gds.graph.filter(
    $filtered_graph_name,
    $base_graph_name,
  '(n:Event AND n.eventTypeId = $event_id)',
  '*',
  { parameters: { event_id: $event_id } }
)
YIELD graphName, fromGraphName, nodeCount, relationshipCount
"""
gds.run_cypher(query_filter_subgraph, params={'base_graph_name': 'graphName','filtered_graph_name': '_graph_name', 'event_id': 1}) 

In [ ]:
gds=get_gdsConnection()
#ratios = {'bombing': 0.001}
ratios = {'bombing': 1, 'shooting': 1, 'earthquake': 1, 'typhoon': 1, 'flood': 1, 'wildfire': 1}
relationshipTypes_whole = ['IS_ABOUT','TALKS_ABOUT', 'MENTIONS', 'POSTED', 'REPLY_TO', 'RETWEETS', 'REPLIED_TO']
eventTypeId = {'bombing': 1, 'shooting': 2, 'earthquake': 3, 'typhoon': 4, 'flood': 5, 'wildfire': 6}
embedding_dim = 128

properties = []

# Construction Dynamique des Propriétés
#drop_wholeGraph(gds) 

for event_type, ratio in ratios.items():
    # Étape Naming subgraph
    subgraph_name = f"{event_type}_subgraph"
    pageRank_name = f"{event_type}_pageRank"
    community_name= f"{event_type}_community"
    dregree_name= f"{event_type}_degree"
    betweeness_name= f"{event_type}_betweeness"
    eigenvector_name= f"{event_type}_eigenvector"
    filtered_graph_name = f"filtered_{event_type.capitalize()}_graph"
    sampled_graph_name = f"sampled_{event_type.capitalize()}_graph"
    
    # When creating properties list for GDS write operation
    subgraph_properties = [f'{pageRank_name}',f'{community_name}', f'{dregree_name}', f'{betweeness_name}',f'{eigenvector_name}']
    # Initialize properties with default values
    init_properties = [f'n.{pageRank_name}=-1',f'n.{community_name}=-1', f'n.{dregree_name}=-1',f'n.{betweeness_name}=-1',f'n.{eigenvector_name}=-1']

    query_subgraph_properties = f"""
        MATCH (n)
        SET {', '.join(init_properties)}
        """

    # crée et initialise query_subgraph_properties en base Neo4j 
    gds.run_cypher(query_subgraph_properties)   
        
    # Étape Create subgraph
    gds.graph.drop(subgraph_name)
    #gds.run_cypher(query_create_main_graph, params={'graphName': subgraph_name, 'properties': properties}) 
    gds.run_cypher(query_create_main_graph, params={'graphName': subgraph_name}) 
    gds.run_cypher(query_filter_subgraph, params={
                                'filtered_graph_name': filtered_graph_name, 
                                'base_graph_name': subgraph_name, 
                                'event_id': eventTypeId[event_type]
                                          })
    #gds.run_cypher(query_filtersubgraph, params={
    #                            'filtered_graph_name': filtered_graph_name, 
    #                            'base_graph_name': subgraph_name, 
    #                            'event_id': eventTypeId[event_type]
    #                                      }) 
    # Étape reduce subgraph
    #gds.run_cypher(query_random_walk, params={
    #                            'sampled_graph_name': sampled_graph_name,
    #                            'subgraph_name': filtered_graph_name,
    #                            'sampling_ratio':ratio, 
    #                            'nodeProperties': subgraph_properties
    #                                    })
    #gds.run_cypher(query_drop, params={'graphName': filtered_graph_name}) 


    #gds.run_cypher(query_drop, params={'graphName': subgraph_name})

    # Calcul des métriques 
    gds.run_cypher(query_pageRank_mutate, params={
                                'graphName': filtered_graph_name, 
                                'pageRank': pageRank_name,
                                'relationshipTypes' :relationshipTypes_whole
                                      })
    #gds.run_cypher(query_drop, params={'graphName': filtered_graph_name})

    gds.run_cypher(query_labelPropagation_mutate, params={'graphName': filtered_graph_name, 'community': community_name})
    gds.run_cypher(degree_query_mutate, params={'graphName': filtered_graph_name, 'degree': dregree_name})
    gds.run_cypher(betweenness_query_mutate, params={'graphName': filtered_graph_name, 'betweenness':betweeness_name}) 
    gds.run_cypher(eigenvector_query_mutate, params={'graphName': filtered_graph_name, 'eigenvector':eigenvector_name }) 
    gds.run_cypher(neo4j_writeNodesProperties, params={'graphName': filtered_graph_name, 'propertiesList': subgraph_properties}) 

    #gds.run_cypher(query_drop, params={'graphName': sampled_graph_name}) 

In [ ]:
#gds.run_cypher("""CALL gds.graph.list() YIELD graphName, schema RETURN distinct graphName""")

gds.run_cypher("""CALL gds.graph.list() YIELD graphName, nodeCount RETURN graphName, nodeCount ORDER BY nodeCount DESC""")


In [ ]:
# Most of the time, we start the graph analysis by running the (weakly) connected components algorithm to get an idea of how (dis)connected our graph really is.
query_weakly_connected_component=f"""
       CALL gds.wcc.stream($graphName) YIELD nodeId, componentId
       RETURN componentId, count(*) as size, 
       collect(gds.util.asNode(nodeId).id) as ids
       ORDER BY size DESC LIMIT 10
       """
gds.run_cypher(query_weakly_connected_component, params={'graphName': 'filtered_Bombing_graph'})

In [ ]:
# Liste les composantes du sous graphe projeté définit
gds.run_cypher("""CALL gds.graph.list('filtered_Bombing_graph') YIELD schema RETURN schema""")

In [ ]:
gds.run_cypher(query_listLabelAttributs)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors
from neo4j_viz.gds import from_gds

def display_Graph(sampled_Graph, nodeProperties, size_property):   
    # 1. Créer une connexion à Neo4j
    gds = get_gdsConnection()

    graph_obj = gds.graph.get(sampled_Graph)
    nodes = gds.graph.nodeProperties.stream(graph_obj,[nodeProperties])
    # 2. Créer un gradient de couleurs personnalisé
    clustering_values = nodes['propertyValue'].values
    # 3. Normaliser les valeurs pour correspondre à la plage de couleurs
    normalize = plt.Normalize(vmin=0, vmax=1)
    colormap = plt.cm.plasma  # Palette continue sans limite de couleurs
    new_colors = [matplotlib.colors.rgb2hex(colormap(normalize(value))) for value in clustering_values]

    VG = from_gds(
        gds,
        graph_obj,
        size_property=size_property,
        additional_node_properties=[nodeProperties],
    )
    VG.color_nodes(nodeProperties, new_colors)
    VG.render(width="100%", height="600px", max_allowed_nodes=11000)

 
display_Graph('filtered_Typhoon_graph','typhoon_degree', 'typhoon_pageRank' )


In [ ]:
gds.run_cypher("""CALL gds.graph.list('filtered_graph_4') YIELD schema RETURN schema""")

__`Interprétation des résultats`__   
 - Un coefficient élevé indique que les voisins d’un nœud sont fortement interconnectés, ce qui est typique pour des nœuds situés au cœur d’un groupe cohésif.

 - À l’inverse, un coefficient faible met en évidence des nœuds dont les voisins ne se connectent pas entre eux, signe qu’ils agissent souvent comme des ponts ou intermédiaires entre des sous-structures distinctes du graphe.
 
- Pour une analyse approfondie, il peut être intéressant de combiner ces résultats avec d’autres métriques, comme la centralité d’intermédiarité, qui souligne également le rôle de connecteur entre différentes communautés

Le diagramme montre une distribution révélatrice des coefficients de clustering local, on  distingue: 

 - Un grand groupe de nœuds avec des coefficients très élevés (proche ou égal à 1), indiquant des sous-réseaux fortement interconnectés où presque tous les voisins sont connectés entre eux. Ces nœuds font probablement partie de communautés cohésives ou de cliques.

 - Une zone intermédiaire (coefficients entre 0.3 et 0.7) qui représente des nœuds ayant un rôle de passerelle partielle, ils sont au sein de communautés mais maintiennent également des connexions avec d’autres groupes.

 - Une bon nombre de nœuds avec des coefficients très faibles (proches de 0), ce qui est particulièrement intéressant pour notre recherche d’intermédiaires entre groupes. Ces nœuds ont des voisins qui ne se connaissent généralement pas entre eux, suggérant qu’ils servent effectivement de ponts entre différentes communautés du réseau.

Cette distribution est typique des réseaux sociaux et organisationnels, où quelques nœuds fortement groupés coexistent avec de nombreux nœuds jouant un rôle d’interface entre communautés.     

Pour identifier précisément les intermédiaires les plus importants, je vous suggère de combiner cette analyse avec d’autres mesures comme la centralité d’intermédiarité (betweenness centrality) et d’examiner spécifiquement les nœuds ayant un faible coefficient de clustering mais un degré relativement élevé.

Un nœud marqué comme point d’articulation signifie qu’il joue un rôle essentiel dans le maintien de la connectivité du graphe.

 - La présence d’un grand nombre de points d’articulation peut indiquer que le réseau est structuré autour de quelques nœuds critiques, dont la défaillance pourrait fragmenter le système global.

 - En pratique, connaître ces nœuds permet d’identifier des vulnérabilités dans divers contextes et d’envisager des mesures pour renforcer la résilience du réseau (par exemple en créant des redondances).

L'objectif est d’identifier des nœuds structurants pour ensuite pouvoir interpréter leurs scores de centralité ou les comparer à des embeddings, il peut être judicieux de lancer gds.articulationPoints avant d’exécuter PageRank ou les méthodes d’embeddings.

In [ ]:
get_number_labelPropagationClst = f"""
    CALL gds.labelPropagation.stream( $graphName )
    YIELD nodeId, communityId AS Community
    RETURN gds.util.asNode(nodeId).name AS Name, Community
    ORDER BY Community, Name
        """
df = pd.DataFrame() 
result = gds.run_cypher(query_inspect_graph)
for index, row in result.iterrows():
    data=gds.run_cypher(get_number_labelPropagationClst, params={'graphName': row['graphName']})
    df = pd.concat([df, data], ignore_index=True)
df 

#### `Comparaison des centralités`

|Mesure	|Portée	|Force|
|:-|:-|:-|
|Eigenvector|	Influence indirecte via le réseau	|Capte les "connexions prestigieuses"|
|Degré|	Connexions directes	|Simple mais superficiel|
|Betweenness|	Contrôle des flux|	Identifie les ponts critiques|

##### __Centralité eigenvector__

La centralité eigenvector repose sur le fait qu'un nœud est influent s'il est connecté à d'autres nœuds influents. Contrairement à la simple centralité de degré, elle capture les dynamiques de pouvoir indirectes dans un réseau.

In [ ]:
eigenvector_query_stream=f"""
    CALL gds.eigenvector.stream($graphName)
    YIELD nodeId, score
    RETURN 
          labels(gds.util.asNode(nodeId)) as label,
          COALESCE(gds.util.asNode(nodeId).screen_name,
          gds.util.asNode(nodeId).id,
          gds.util.asNode(nodeId).topic) AS name,
          score
    ORDER BY score DESC, name ASC
        """

df = pd.DataFrame() 
result = gds.run_cypher(query_inspect_graph)
for index, row in result.iterrows():
    data=gds.run_cypher(eigenvector_query_stream, params={'graphName': row['graphName']})
    df = pd.concat([df, data], ignore_index=True)
df    

In [ ]:
degree_query_stream = f"""
    CALL gds.degree.stream($graphName)
    YIELD nodeId, score
    RETURN Labels(gds.util.asNode(nodeId)) AS Label,
           gds.util.asNode(nodeId).id as id, 
           COALESCE(gds.util.asNode(nodeId).name,
                    gds.util.asNode(nodeId).screen_name,
                    gds.util.asNode(nodeId).eventType) AS name, 
           score AS followers     
    ORDER BY followers DESC, name DESC
    """

df = pd.DataFrame() 
result = gds.run_cypher(query_inspect_graph)
for index, row in result.iterrows():
    data=gds.run_cypher(degree_query_stream, params={'graphName': row['graphName']})
    df = pd.concat([df, data], ignore_index=True)
df    

In [ ]:
betweenness_query_stream = f"""
    CALL gds.betweenness.stream($graphName, {{samplingSize: 2, samplingSeed: 4}})
    YIELD nodeId, score
    RETURN 
          labels(gds.util.asNode(nodeId)) as label,
          COALESCE(gds.util.asNode(nodeId).screen_name,
          gds.util.asNode(nodeId).id,
          gds.util.asNode(nodeId).topic) AS name,
          score
    ORDER BY score desc""" 

df = pd.DataFrame() 
result = gds.run_cypher(query_inspect_graph)
for index, row in result.iterrows():
    data=gds.run_cypher(betweenness_query_stream, params={'graphName': row['graphName']})
    df = pd.concat([df, data], ignore_index=True)
df    

In [ ]:
# vérification des résultats du page_ranking
query_pageRank_stream=  f""" 
    CALL gds.pageRank.stream($graphName)
    YIELD nodeId, score
    RETURN LABELS(gds.util.asNode(nodeId)) AS Label,
           COALESCE(gds.util.asNode(nodeId).eventType,gds.util.asNode(nodeId).screen_name) AS Event,
           gds.util.asNode(nodeId).id as id,
           COALESCE(gds.util.asNode(nodeId).trecisid,gds.util.asNode(nodeId).topic,gds.util.asNode(nodeId).name ) AS Topic,
           score as Score 
    ORDER BY score DESC
    """

df = pd.DataFrame() 
result = gds.run_cypher(query_inspect_graph)
for index, row in result.iterrows():
    data=gds.run_cypher(query_pageRank_stream, params={'graphName': row['graphName']})
    df = pd.concat([df, data], ignore_index=True)
df


|Étape	  |Procédure GDS  |	Objectif  |
|---        |:-:        |:-:            |
|Graph Neural Networks  |	gds.beta.graphDataScience  |	Utiliser des réseaux de neurones  |
|Graph Convolutional Networks  |	gds.beta.graphDataScience  |	Utiliser des réseaux de neurones  |
|Graph Attention Networks  |	gds.beta.graphDataScience  |	Utiliser des réseaux de neurones  |
|Graph Autoencoders  |	gds.beta.graphDataScience  |	Utiliser des réseaux de neurones  |
|Graph Recurrent Networks  |	gds.beta.graphDataScience  |	Utiliser des réseaux de neurones  |

In [ ]:
veriffastRP_query=  f""" 
CALL gds.fastRP.stream(
  $graphName,
  {{
    embeddingDimension: 128,   
    iterationWeights: [1,1,1,1,1 ],
    normalizationStrength: 0.01,
    propertyRatio: 0,
    randomSeed: 42,
    nodeLabels: ['User','Event', 'Tweet', 'Hashtag', 'PostCategory' ] ,
    relationshipTypes: ['HAS_HASHTAG', 'MENTIONS', 'POSTED', 'REPLIED_TO', 'REPLY_TO', 'RETWEETS']   
  }}
)
YIELD nodeId, embedding
RETURN embedding,  
  CASE 
    WHEN gds.util.asNode(nodeId):Hashtag THEN 'Event'
    WHEN gds.util.asNode(nodeId):User THEN 'User'
    WHEN gds.util.asNode(nodeId):Tweet THEN 'Tweet'
    WHEN gds.util.asNode(nodeId):Hashtag THEN 'Hashtag'
    WHEN gds.util.asNode(nodeId):PostCategory THEN 'PostCategory'
  END AS Label,
    COALESCE(gds.util.asNode(nodeId).eventType,gds.util.asNode(nodeId).name) AS Event,
    gds.util.asNode(nodeId).id as id,
    gds.util.asNode(nodeId).eventType AS eventType,
    COALESCE(gds.util.asNode(nodeId).trecisid,gds.util.asNode(nodeId).topic,gds.util.asNode(nodeId).name ) AS Topic
ORDER BY Label
"""
df = pd.DataFrame() 
result = gds.run_cypher(query_inspect_graph)
for index, row in result.iterrows():
    print(row)
    data=gds.run_cypher(veriffastRP_query, params={'graphName': row['graphName']})
    df = pd.concat([df, data], ignore_index=True)
df

<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=500 large=400/>
</p>

__`Recommend (without a query)`__    

  – Content: blog, web page, wiki, map, ...    
  – People: person to fellow, collaborator, friend, ...    
  – Item: job, product,...

__`Retrieve (with a query)`__  

  – Content: answer to a question, document, comment,    
  – People: person based on topics, opinions, ..    
  – Items : products with characteristics, job with skills, ...    

__`Detect`__     

  – Anormal behaviors (anomalies)
  – Events
  –
...

__`Identify`__
- Influencers
- Trending topics    
...


 :::

__`Graph-based description of social recommendation`__ 
   
 - __The bi-partite graph__ of traditional recommendation `user-user interactions are not considered`     
 - __The heterogeneous graph__ of traditional recommendation `user-user interactions are considered`

:::

:::

__`Main tasks of recommender systems`__   

 - __The typical tasks of traditional recommendation__ : User-Item Ratings (ne s'applique pas à notre cas).      
 - __Rating prediction__: predict the rating of that user u will give to his or her unrated item i. 
     - `considering, ratings are available`. 
     - a typical regression or (multi-class) classification problem.          
 - __Top N recommendation__: 
     - `considering ratings are not available`, only the items the user purchased, rating prediction is not possible.

 :::